# Guitar-to-MIDI TinyML Pipeline -- Raw-Audio Dual-Head DS-CNN

This notebook trains, quantizes, and exports the note-detection model used by the ESP32-S3
guitar-to-MIDI firmware (`guitar_midi_esp32s3.ino`). It turns a single-channel guitar audio
stream into two synchronized predictions per analysis window:

- **Note** -- which pitch (or silence) is currently sounding, as one class in a data-derived
  MIDI vocabulary.
- **Onset** -- whether a new note attack just occurred, which the firmware uses to decide
  exactly when to emit a MIDI Note On.

## Pipeline at a glance

1. **Dataset** -- [GuitarSet](https://guitarset.weebly.com/) solo recordings, single
   reference mic, annotated per-string note transcriptions.
2. **Framing** -- 128 ms sliding analysis windows (2048 samples @ 16 kHz), 10 ms hop, labels
   referenced to the causal (streaming) end of the window.
3. **Model** -- a MobileNet-style depthwise-separable CNN over raw PCM, sized to fit the
   firmware's real-time inference budget on the ESP32-S3 (see "Model capacity vs. inference
   budget" below).
4. **Quantization** -- INT8 via quantization-aware training (QAT), verified against a
   checkpoint ladder (float32 -> dynamic-range -> PTQ int8 -> QAT int8) so every stage of the
   conversion is measured rather than assumed.
5. **Export** -- `model.h` (the quantized model as a C array) and `class_labels.h` (every
   constant the firmware must agree with this notebook on), copied straight into the Arduino
   sketch folder.

## Running this notebook

```
pip install -r requirements.txt
jupyter lab train_guitar_midi.ipynb
```

Point `GUITARSET_DIR` (in the CONFIG cell) at a local copy of GuitarSet containing
`audio_mono-mic/` and `annotation/`, then run top to bottom. Outputs land in `./artifacts/`:

- `guitar_midi_model.keras` -- the trained float model
- `final_audio_model_int8.tflite` -- the deployed quantized model
- `model.h`, `class_labels.h` -- headers for the firmware
- `run_summary.json` -- machine-readable metrics for every checkpoint

Copy `model.h` and `class_labels.h` into the same folder as `guitar_midi_esp32s3.ino` before
compiling the firmware in the Arduino IDE. See `README.md` at the project root for how the
notebook and firmware fit together end to end.

> **TensorFlow / Keras version note.** `tensorflow_model_optimization` (used for the QAT
> checkpoint below) checks model types against **Keras 2** internals. TF 2.16+ defaults to
> Keras 3, whose `Model` class doesn't expose what `tfmot` looks for, so `quantize_model()`
> raises `` `to_quantize` can only either be a keras Sequential or Functional model. `` even on
> a valid functional model, simply because it is a *Keras 3* functional model.
>
> The cell below binds directly to the `tf_keras` package (Keras 2) rather than relying on the
> `TF_USE_LEGACY_KERAS` environment variable, since that variable only takes effect if nothing
> else in the kernel has imported TensorFlow yet -- easy to violate across kernel restarts or
> out-of-order cell execution. `requirements.txt` lists `tf_keras` explicitly so this works out
> of the box. If you hit the Keras-3 error anyway on an existing environment: `pip install
> tf_keras`, then **restart the kernel** (a bare `pip install` doesn't un-import an
> already-imported `tensorflow`) and re-run from the top.

In [ ]:
import os

# Belt-and-suspenders: harmless if the direct tf_keras import below succeeds (which is the
# path that actually matters), but keeps this notebook working even in environments where
# tf_keras is unavailable but TF_USE_LEGACY_KERAS is otherwise honoured.
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")

import glob
import hashlib
import json
import math
import pathlib
import sys
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd
import scipy.signal
import tensorflow as tf

# --- Bind to a Keras-2-compatible `keras`, robustly (see the markdown above) ---
# Import tf_keras BY NAME rather than going through tf.keras + an env var, so this doesn't
# depend on import order within the kernel. layers/models are taken from the SAME module as
# keras, guaranteeing class identity matches what tensorflow_model_optimization expects.
try:
    import tf_keras as keras
    _KERAS_SOURCE = "tf_keras (explicit Keras 2 binding)"
except ImportError:
    from tensorflow import keras
    _KERAS_SOURCE = "tensorflow.keras (tf_keras not installed -- fell back to whatever this TF ships)"

layers = keras.layers
models = keras.models

if keras.__version__.startswith("3."):
    raise RuntimeError(
        f"Bound to Keras {keras.__version__} via {_KERAS_SOURCE}, but "
        "tensorflow_model_optimization (the QAT checkpoint below) does not support Keras 3.\n\n"
        "Fix:  pip install tf_keras\n"
        "      then RESTART THE KERNEL (a pip install alone does not un-import an\n"
        "      already-imported tensorflow) and re-run this notebook from the top.\n\n"
        "Alternative:  pip install \"tensorflow>=2.15,<2.16\" -- a TF build where Keras 2 is\n"
        "      the default and no shim is needed at all."
    )
print(f"Keras binding: {_KERAS_SOURCE} -- version {keras.__version__}")

import matplotlib.pyplot as plt

try:
    import librosa
except ImportError:
    raise SystemExit(
        "librosa is required for robust WAV loading/resampling.\n"
        "Install with: pip install librosa soundfile"
    )

# A small fixed categorical/sequential palette, used consistently across every plot in this
# notebook instead of matplotlib defaults.
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95"]
SURFACE, INK, INK2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": GRID,
    "axes.labelcolor": INK2, "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
})

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version:      {keras.__version__}")

## Dataset: GuitarSet solo recordings (real annotated performances)

[GuitarSet](https://guitarset.weebly.com/) (Xi, Bittner, Pauwels, Ye & Bello, ISMIR 2018):
360 continuous ~30s performances recorded on a real acoustic guitar, each with a rich
`.jams` annotation file including per-string note transcriptions, pitch contours, beats,
tempo, and chords.

**What's used here, and why:**

- **Audio: `audio_mono-mic/` only** -- the single reference-mic recording, not the
  6-channel hexaphonic pickup (`audio_hex`/`audio_hex_cln`). The ESP32 firmware's primary
  input is one INMP441 mic, so training on the same single-channel signal it will actually
  hear avoids a train/deploy mismatch.
- **Solo takes only, not comping** -- GuitarSet records each of its 30 lead sheets twice
  per player: a comping (chords/rhythm) pass and a solo (single-note lead line) pass. This
  model's note head is an *N*-way single-note softmax, which assumes monophonic content;
  comping is chordal and would violate that assumption throughout.
- **Annotations parsed directly from `.jams` (plain JSON)**, not the `jams` Python package
  -- one less dependency, and full control over how six per-string monophonic annotations
  get reduced to one label per frame.

**Design decisions specific to this pipeline:**

1. **The note-class range is data-driven**, not a hardcoded MIDI range. Solo lead lines
   routinely climb well past the guitar's open-position notes. `MIDI_HIGH` is read off the
   corpus, capped at a high percentile (`MIDI_HIGH_PERCENTILE`) so a handful of extreme
   outlier notes don't add softmax classes that have five training examples each.
2. **The train/val/test split is by *player*, not by file.** Splitting at the file level
   lets the same guitarist's tone and technique appear in both train and test, which
   inflates apparent accuracy. One player is held out entirely, which is also GuitarSet's
   own recommended evaluation protocol.
3. **Frame labels come from real annotated note timings**, rasterized from each note's
   `time`/`duration` across all 6 strings.

## CONFIG -- edit these to taste

Anything in this cell that the ESP32 firmware also needs is exported automatically into
`class_labels.h` at the end of the notebook (window size, hop size, normalization floor,
onset window, class table, onset threshold), so the two can't silently drift apart.

In [ ]:
GUITARSET_DIR = pathlib.Path(r"C:\AREAS\skool\Q4\ee446\final project\datasets\guitarset")
AUDIO_DIR = GUITARSET_DIR / "audio_mono-mic"   # mono reference-mic recordings only (no hex / hex_cln)
ANNOTATION_DIR = GUITARSET_DIR / "annotation"  # matching .jams files

OUT_DIR = pathlib.Path("artifacts")
OUT_DIR.mkdir(exist_ok=True)

SAMPLE_RATE = 16000

# --- Framing ---------------------------------------------------------------
# MUST match WINDOW_SAMPLES / HOP_SAMPLES in the ESP32 firmware exactly. Both are re-exported
# into class_labels.h at the end of this notebook, and the firmware #errors at compile time if
# either constant is missing, so a mismatch is caught immediately rather than as a silent
# accuracy loss on-device.
#
# 2048 samples = 128 ms gives ~7.8 Hz of frequency resolution, comfortably resolving the
# 4.9 Hz semitone gap between the guitar's lowest two notes (E2/F2). STEM_STRIDE (below) keeps
# the cost of the trunk downstream of the stem independent of this choice.
WINDOW_SAMPLES = 2048
HOP_SAMPLES = 160       # 10 ms stride between successive frames

# Stem convolution stride. Halving the timesteps the stem emits halves the cost of every layer
# downstream of it -- linear and quadratic terms alike -- with no change to channel counts. The
# cheapest lever for fitting the model inside the firmware's real-time inference budget; see
# "Model capacity vs. inference budget" below for how this value was chosen alongside MODEL_WIDTH.
STEM_STRIDE = 8

# --- Note vocabulary -------------------------------------------------------
MIDI_LOW = 40                 # guitar's physical floor in standard tuning (open low E)
MIDI_HIGH_PERCENTILE = 99.9   # cap the top of the vocabulary here rather than at the
                              # absolute max, so rare outliers don't create near-empty classes

# --- Label timing ----------------------------------------------------------
# Labels are referenced to the END of the window (streaming/causal convention), backed off by
# LABEL_LAG_MS -- see "Step 4" below for the full rationale. The small backoff exists because
# in the first few ms after an attack the pitch genuinely isn't determinable yet, so labelling
# the very newest sample would just inject label noise.
LABEL_LAG_MS = 15.0

# A window is an "onset" window if a real annotated attack falls inside the last
# ONSET_WINDOW_MS of it. At a 10 ms hop that gives each note ~6 consecutive onset-positive
# frames -- wide enough that the firmware's inference loop cannot step over the entire positive
# band even if a single inference takes several hop periods, which is what makes the onset
# head's timing robust to inference speed rather than merely fast at one particular speed.
ONSET_WINDOW_MS = 60

# Any window whose RMS falls below this (dBFS) is relabelled silence regardless of what the
# annotation says -- catches natural note-decay tails and annotation/audio mismatches.
SILENCE_RMS_DBFS = -50.0

# --- Window normalization ---------------------------------------------------
# Every window is divided by max(peak, NORM_PEAK_FLOOR) so that any window at or above the
# floor lands with peak exactly 1.0 and therefore uses the full int8 input range. Windows below
# the floor are amplified by at most 1/NORM_PEAK_FLOOR and stay under 1.0, which preserves
# loudness as a usable cue for the silence class instead of amplifying room noise to full scale.
#
# -40 dBFS peak sits roughly 10 dB above the -50 dBFS RMS silence gate, i.e. just above the
# peak level of a window that is about to be relabelled silence anyway. THIS FLOOR, AND THE
# NORMALIZATION FORMULA ITSELF, MUST MATCH quantize_and_load_input() IN THE FIRMWARE EXACTLY --
# see the "normalization parity check" cell below.
NORM_PEAK_FLOOR_DBFS = -40.0
NORM_PEAK_FLOOR = 10.0 ** (NORM_PEAK_FLOOR_DBFS / 20.0)   # 0.01

# --- Splits ----------------------------------------------------------------
TEST_PLAYERS = {"05"}   # held out ENTIRELY from train/val
VAL_FRACTION = 0.2      # of the remaining players' files

# --- Class balance ---------------------------------------------------------
# Silence dominates the frame distribution, which both inflates headline accuracy and biases
# the softmax. Training frames labelled silence are randomly dropped down to this fraction.
# Evaluation is NEVER downsampled -- test/val see the true distribution.
MAX_SILENCE_FRACTION = 0.15
# Per-class sample weights for the note head, w ~ 1/sqrt(freq), clipped to this range.
CLASS_WEIGHT_CLIP = (0.25, 4.0)

# --- Augmentation (train split only, applied after the cache) --------------
AUGMENT = True
AUG_GAIN_DB = (-9.0, 6.0)     # random level, matters relative to NORM_PEAK_FLOOR
AUG_SNR_DB = (15.0, 45.0)     # additive white noise at this SNR range
AUG_POLARITY_FLIP = True      # a pickup/mic can be wired either way; the model shouldn't care

# --- Model capacity ----------------------------------------------------------
# Scales every filter count (see _w() in the model section). Chosen together with STEM_STRIDE
# above so the model fits the firmware's ~20 ms real-time inference budget on the ESP32-S3:
# at this configuration the model costs ~1.44 MMAC/window against a measured on-device
# throughput of ~91.4 MMAC/s (esp-nn INT8 kernels) -- see "Model capacity vs. inference budget"
# below for the full sweep this was chosen from.
MODEL_WIDTH = 0.5
DROPOUT = 0.2

# --- Training --------------------------------------------------------------
BATCH_SIZE = 128
EPOCHS = 60
LAMBDA_ONSET = 2.0          # weight on the onset focal-BCE term in the combined loss
LEARNING_RATE = 2e-3
QAT_EPOCHS = 8
QAT_LEARNING_RATE = 1e-4
SEED = 42

# --- Cache -----------------------------------------------------------------
FORCE_REBUILD_CACHE = False   # set True to ignore any existing on-disk tf.data cache

tf.random.set_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

hop_seconds = HOP_SAMPLES / SAMPLE_RATE
window_seconds = WINDOW_SAMPLES / SAMPLE_RATE
onset_window_s = ONSET_WINDOW_MS / 1000.0
label_lag_s = LABEL_LAG_MS / 1000.0

print(f"Window: {WINDOW_SAMPLES} samples = {window_seconds * 1000:.0f} ms "
      f"(frequency resolution ~{SAMPLE_RATE / WINDOW_SAMPLES:.1f} Hz)")
print(f"Hop:    {HOP_SAMPLES} samples = {hop_seconds * 1000:.0f} ms")
print(f"Stem stride {STEM_STRIDE} -> trunk sees {WINDOW_SAMPLES // STEM_STRIDE} timesteps")
print(f"Normalization floor: {NORM_PEAK_FLOOR_DBFS:.0f} dBFS peak = {NORM_PEAK_FLOOR:.4f}")

# Sanity: semitone spacing at the bottom of the range vs. what the window can resolve.
_f_low = 440.0 * 2 ** ((MIDI_LOW - 69) / 12)
_f_next = 440.0 * 2 ** ((MIDI_LOW + 1 - 69) / 12)
print(f"At MIDI {MIDI_LOW}: {_f_low:.2f} Hz -> {_f_next:.2f} Hz, a {_f_next - _f_low:.2f} Hz step "
      f"vs. {SAMPLE_RATE / WINDOW_SAMPLES:.1f} Hz of window resolution "
      f"({'OK' if (_f_next - _f_low) > SAMPLE_RATE / WINDOW_SAMPLES else 'relies on harmonics'})")

## Step 1 -- Discover solo recordings and match them to annotations

In [ ]:
def discover_solo_files(audio_dir: pathlib.Path, annotation_dir: pathlib.Path) -> list:
    """Pairs every *_solo_mic.wav with its matching *_solo.jams annotation.

    The "_comp_" filter is a defensive no-op against this copy of GuitarSet (which is
    already solo-only) in case a fuller download with comping takes ever replaces it.
    """
    if not audio_dir.is_dir():
        raise FileNotFoundError(
            f"Audio directory not found: {audio_dir}\n"
            "Point GUITARSET_DIR at the folder that contains audio_mono-mic/ and annotation/."
        )
    records = []
    skipped_comp, skipped_no_annotation = 0, 0
    for wav_path in sorted(audio_dir.glob("*_mic.wav")):
        if "_comp_" in wav_path.name:
            skipped_comp += 1
            continue
        if "_solo_" not in wav_path.name:
            continue
        stem = wav_path.name[: -len("_solo_mic.wav")]  # e.g. "00_BN1-129-Eb"
        jams_path = annotation_dir / f"{stem}_solo.jams"
        if not jams_path.exists():
            skipped_no_annotation += 1
            continue
        records.append({
            "wav": wav_path,
            "jams": jams_path,
            "player": stem.split("_")[0],   # "00".."05"
            "stem": stem,
        })
    print(f"Discovered {len(records)} solo recordings with matching annotations "
          f"(skipped {skipped_comp} comp takes, {skipped_no_annotation} missing annotations)")
    print(f"Players present: {sorted({r['player'] for r in records})}")
    return records


records = discover_solo_files(AUDIO_DIR, ANNOTATION_DIR)
if not records:
    raise RuntimeError(
        "No solo recordings found. Point GUITARSET_DIR at the folder that "
        "contains audio_mono-mic/ and annotation/ from the GuitarSet download."
    )

## Step 2 -- Parse the JAMS `note_midi` annotations and derive the note vocabulary from the data

In [ ]:
def load_jams_note_events(jams_path: pathlib.Path):
    """Parses the 6 per-string `note_midi` namespaces out of a JAMS file into one flat,
    time-sorted list of (start_s, end_s, string_idx, midi_pitch) tuples. JAMS is plain
    JSON, so no extra dependency is needed -- annotation_metadata.data_source gives the
    string index (0 = low E, 5 = high e), matching GuitarSet's convention.
    """
    with open(jams_path) as f:
        data = json.load(f)
    events = []
    for ann in data["annotations"]:
        if ann["namespace"] != "note_midi":
            continue
        string_idx = ann["annotation_metadata"].get("data_source")
        try:
            string_idx = int(string_idx)
        except (TypeError, ValueError):
            string_idx = -1
        for note in ann["data"]:
            start = float(note["time"])
            duration = float(note["duration"])
            if duration <= 0:
                continue
            events.append((start, start + duration, string_idx, float(note["value"])))
    events.sort(key=lambda e: e[0])
    file_duration = float(data["file_metadata"]["duration"])
    return events, file_duration


print("Parsing JAMS annotations for every discovered recording...")
per_file_events = {}     # stem -> (events, file_duration)
all_pitches, all_durations = [], []
notes_per_string = {i: 0 for i in range(6)}
notes_per_player = {}
for rec in records:
    events, file_duration = load_jams_note_events(rec["jams"])
    per_file_events[rec["stem"]] = (events, file_duration)
    notes_per_player[rec["player"]] = notes_per_player.get(rec["player"], 0) + len(events)
    for (start, end, string_idx, pitch) in events:
        all_pitches.append(pitch)
        all_durations.append(end - start)
        if string_idx in notes_per_string:
            notes_per_string[string_idx] += 1

all_pitches = np.array(all_pitches)
all_durations = np.array(all_durations)
print(f"Total annotated notes across {len(records)} recordings: {len(all_pitches):,}")

# --- Data-driven note vocabulary -------------------------------------------
# MIDI_LOW (40, open low E) is fixed -- it's the guitar's physical floor. MIDI_HIGH is read
# off the corpus, but at a high PERCENTILE rather than the absolute max: taking the max
# lets a handful of extreme outlier notes append softmax classes that have only a few
# training frames each, which costs parameters and adds confusable classes for no benefit.
MIDI_HIGH = int(np.ceil(np.percentile(all_pitches, MIDI_HIGH_PERCENTILE)))
n_below_low = int(np.sum(np.round(all_pitches) < MIDI_LOW))
n_above_high = int(np.sum(np.round(all_pitches) > MIDI_HIGH))
print(f"MIDI_HIGH from the {MIDI_HIGH_PERCENTILE} percentile of the corpus: {MIDI_HIGH} "
      f"(absolute max in the data: {int(np.ceil(all_pitches.max()))})")
if n_below_low:
    print(f"  {n_below_low} annotated notes ({100 * n_below_low / len(all_pitches):.2f}%) "
          f"fall below MIDI_LOW={MIDI_LOW} and are clipped to it.")
if n_above_high:
    print(f"  {n_above_high} annotated notes ({100 * n_above_high / len(all_pitches):.2f}%) "
          f"fall above MIDI_HIGH={MIDI_HIGH} and are clipped to it.")

class_names = ["silence"] + [str(m) for m in range(MIDI_LOW, MIDI_HIGH + 1)]
num_classes = len(class_names)
label_to_index = {name: idx for idx, name in enumerate(class_names)}
SILENCE_INDEX = label_to_index["silence"]
print(f"Note vocabulary: silence + MIDI {MIDI_LOW}-{MIDI_HIGH} = {num_classes} classes")

### Visualization -- pitch coverage vs. the note-classification head

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
bins = np.arange(min(MIDI_LOW, all_pitches.min()) - 2, all_pitches.max() + 3, 1)
ax.hist(all_pitches, bins=bins, color=SEQ_BLUE[3], edgecolor=SURFACE, linewidth=0.5)
ax.axvline(MIDI_LOW, color=CAT[7], linestyle="--", linewidth=1.5, label="MIDI_LOW (fixed)")
ax.axvline(MIDI_HIGH, color=CAT[1], linestyle="-", linewidth=1.5,
           label=f"MIDI_HIGH ({MIDI_HIGH_PERCENTILE}th pct of the corpus)")
ax.set_xlabel("MIDI pitch"); ax.set_ylabel("annotated note count")
ax.set_title("Pitch distribution across all annotated solo notes")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

pct_clipped_now = 100 * np.mean((np.round(all_pitches) < MIDI_LOW) | (np.round(all_pitches) > MIDI_HIGH))
print(f"{pct_clipped_now:.2f}% of annotated notes fall outside the data-driven "
      f"MIDI {MIDI_LOW}-{MIDI_HIGH} head and are clipped to its edges.")

### Visualization -- per-string usage and note-duration distribution

In [ ]:
string_names = ["E2 (low E)", "A2", "D3", "G3", "B3", "E4 (high e)"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(string_names, [notes_per_string[i] for i in range(6)], color=CAT[:6])
axes[0].set_ylabel("annotated note count")
axes[0].set_title("Notes per string (note_midi annotations)")
axes[0].tick_params(axis="x", rotation=20)

axes[1].hist(np.clip(all_durations, 0, 2.0), bins=60, color=SEQ_BLUE[3], edgecolor=SURFACE, linewidth=0.3)
axes[1].axvline(window_seconds, color=CAT[1], linestyle="--", linewidth=1.5,
                label=f"analysis window ({window_seconds * 1000:.0f} ms)")
axes[1].set_xlabel("note duration (s, clipped at 2s)"); axes[1].set_ylabel("count")
axes[1].set_title(f"Note duration (median {np.median(all_durations) * 1000:.0f} ms)")
axes[1].legend(frameon=False)

fig.tight_layout()
plt.show()

short_notes = float(np.mean(all_durations < window_seconds))
print(f"{short_notes * 100:.1f}% of annotated notes are shorter than the {window_seconds * 1000:.0f} ms "
      f"analysis window -- for those, the window necessarily contains part of a neighbouring note.")

## Step 3 -- Player-held-out train / val / test split

In [ ]:
players = sorted({r["player"] for r in records})
missing_test_players = TEST_PLAYERS - set(players)
if missing_test_players:
    print(f"WARNING: TEST_PLAYERS {missing_test_players} not found in the discovered recordings")

train_val_players = [p for p in players if p not in TEST_PLAYERS]
print(f"Players: {players}")
print(f"Held-out TEST player(s) (never seen in train/val): {sorted(TEST_PLAYERS)}")
print(f"Train/val players: {train_val_players}")

test_files = [r for r in records if r["player"] in TEST_PLAYERS]
train_val_files = [r for r in records if r["player"] in train_val_players]

perm = rng.permutation(len(train_val_files))
val_count = int(len(perm) * VAL_FRACTION)
val_idx, train_idx = perm[:val_count], perm[val_count:]
train_files = [train_val_files[i] for i in train_idx]
val_files = [train_val_files[i] for i in val_idx]

print(f"Files -- train: {len(train_files)}  val: {len(val_files)}  test: {len(test_files)}")
for split_name, split_files in [("train", train_files), ("val", val_files), ("test", test_files)]:
    total_notes = sum(len(per_file_events[r["stem"]][0]) for r in split_files)
    total_seconds = sum(per_file_events[r["stem"]][1] for r in split_files)
    print(f"  {split_name}: {len(split_files)} files, {total_notes:,} notes, "
          f"{total_seconds / 60:.1f} min of audio")

# --- Visualization: split composition per player ---------------------------
split_of_player = {}
for split_name, split_files in [("train", train_files), ("val", val_files), ("test", test_files)]:
    for r in split_files:
        split_of_player.setdefault(r["player"], {"train": 0, "val": 0, "test": 0})[split_name] += 1

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(players))
width = 0.25
for i, (split_name, color) in enumerate([("train", CAT[0]), ("val", CAT[1]), ("test", CAT[7])]):
    vals = [split_of_player.get(p, {}).get(split_name, 0) for p in players]
    ax.bar(x + (i - 1) * width, vals, width, label=split_name, color=color)
ax.set_xticks(x); ax.set_xticklabels([f"player {p}" for p in players])
ax.set_ylabel("file count")
ax.set_title("Train/val/test split by player -- the test player never appears in train or val")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## Step 4 -- Rasterize continuous annotations into frame-level (note, onset) labels

**Label timing convention.** Both labels are referenced to the **end** of the window rather
than its centre -- the standard causal/streaming convention, and the one the firmware's
real-time pipeline actually needs:

```
    window h:  [ hop_start ................................ hop_start + 128 ms ]
                                                    ^                    ^
                                     note label read here          window end
                                     (end - LABEL_LAG_MS)
                                            |<-- ONSET_WINDOW_MS -->|
                                            onset = 1 if a real attack lands in here
```

Referencing the window's centre instead would cost `WINDOW/2` = 64 ms of unavoidable label
lag, which the firmware would then pay for again as note-on latency. The end-referenced
convention lets the onset head fire roughly `ONSET_WINDOW_MS` after a real attack rather than
a full window afterwards.

`LABEL_LAG_MS` backs the note label off slightly from the newest sample because in the first
few milliseconds after an attack the pitch genuinely isn't determinable, so labelling the very
newest sample would only inject label noise.

The onset label depends only on whether a real annotated attack falls in the window's trailing
`ONSET_WINDOW_MS` -- it is *not* conjoined with "a note is currently active", since under the
end-referenced convention the very first positive window for a note-after-a-rest has no note
active yet at all.

In [ ]:
def compute_frame_labels(events, file_duration, hop_seconds, window_seconds,
                         onset_window_s, label_lag_s, midi_low, midi_high, silence_index):
    """Turns a file's note events into two per-hop arrays covering the whole recording:

      note_idx[h]  class index of the note sounding at hop h's LABEL POINT
                   (= window end - label_lag), 0 == silence
      onset[h]     1.0 if some note's real annotated attack falls inside the last
                   onset_window_s of hop h's window

    When multiple strings ring at once (double stops, sympathetic ringing, mic bleed) the
    HIGHEST-pitched active note wins -- the standard monophonic reduction for lead lines.
    """
    n_hops = max(1, int(math.ceil(file_duration / hop_seconds)))
    hop_start = np.arange(n_hops) * hop_seconds
    window_end = hop_start + window_seconds
    label_time = window_end - label_lag_s

    note_idx = np.full(n_hops, silence_index, dtype=np.int32)
    onset = np.zeros(n_hops, dtype=np.float32)
    if not events:
        return note_idx, onset

    starts = np.array([e[0] for e in events])
    ends = np.array([e[1] for e in events])
    pitches = np.clip(np.round([e[3] for e in events]), midi_low, midi_high).astype(int)

    # --- note label: highest-pitched note sounding at the label point ---
    active = (starts[None, :] <= label_time[:, None]) & (label_time[:, None] < ends[None, :])
    any_active = active.any(axis=1)
    masked_pitch = np.where(active, pitches[None, :], -1)
    winner = masked_pitch.argmax(axis=1)
    active_hops = np.nonzero(any_active)[0]
    # class index = 1 + (midi - midi_low); index 0 is reserved for silence.
    note_idx[active_hops] = pitches[winner[active_hops]] - midi_low + 1

    # --- onset label: a real attack inside the window's trailing onset_window_s ---
    onset_lo = (window_end - onset_window_s)[:, None]
    onset_hi = window_end[:, None]
    in_onset_band = (starts[None, :] >= onset_lo) & (starts[None, :] < onset_hi)
    onset[:] = in_onset_band.any(axis=1).astype(np.float32)
    return note_idx, onset


print("Rasterizing every recording's annotations into frame-level labels...")
frame_labels_by_stem = {}
for rec in records:
    events, file_duration = per_file_events[rec["stem"]]
    frame_labels_by_stem[rec["stem"]] = compute_frame_labels(
        events, file_duration, hop_seconds, window_seconds, onset_window_s, label_lag_s,
        MIDI_LOW, MIDI_HIGH, SILENCE_INDEX,
    )

all_note_idx = np.concatenate([frame_labels_by_stem[r["stem"]][0] for r in records])
all_onset = np.concatenate([frame_labels_by_stem[r["stem"]][1] for r in records])
print(f"Total rasterized frames: {len(all_note_idx):,}")
print(f"Silence fraction (before the RMS override in Step 5): {np.mean(all_note_idx == SILENCE_INDEX):.3f}")
onset_pos_fraction = float(np.mean(all_onset > 0.5))
print(f"Onset-positive fraction: {onset_pos_fraction:.4f}")
print(f"  -> a constant-'no onset' predictor scores {100 * (1 - onset_pos_fraction):.2f}% "
      f"'accuracy'. This is exactly why the onset head is scored by precision/recall/F1 "
      f"below and not by accuracy.")

expected_onset_frames = onset_window_s / hop_seconds
print(f"Expected onset-positive frames per note: ~{expected_onset_frames:.0f} "
      f"(observed: {all_onset.sum() / max(1, len(all_pitches)):.2f})")

### Visualization -- class balance, and a sanity-check piano roll for one example clip

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
counts = np.bincount(all_note_idx, minlength=num_classes)
colors = [CAT[7] if i == SILENCE_INDEX else CAT[0] for i in range(num_classes)]
ax.bar(range(num_classes), counts, color=colors, width=1.0)
ax.set_yscale("log")
ax.set_xlabel("class index (0 = silence, then ascending MIDI pitch)")
ax.set_ylabel("frame count (log scale)")
ax.set_title("Class balance across all rasterized frames -- silence in red")
fig.tight_layout()
plt.show()

# Does the derived frame-label sequence actually track the annotated notes? If these two
# panels don't line up, something is wrong upstream of training.
example = train_files[0]
events, file_duration = per_file_events[example["stem"]]
note_idx_seq, onset_seq = frame_labels_by_stem[example["stem"]]
t_max = min(10.0, file_duration)

fig, axes = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True, height_ratios=[3, 2])
ax = axes[0]
for (start, end, string_idx, pitch) in events:
    if start > t_max:
        continue
    color = CAT[string_idx % 6] if string_idx >= 0 else MUTED
    ax.plot([start, min(end, t_max)], [pitch, pitch], color=color, linewidth=4, solid_capstyle="butt")
    ax.plot([start], [pitch], marker="|", color=INK, markersize=10)
ax.set_ylabel("MIDI pitch")
ax.set_title(f"{example['stem']} -- annotated notes (colour = string) vs. derived frame labels, first {t_max:.0f}s")

ax2 = axes[1]
hop_times = np.arange(len(note_idx_seq)) * hop_seconds
mask = hop_times <= t_max
# Plot the derived label against the time it REFERS to (window end - lag), not the hop
# start, so it should overlay the annotation panel above rather than sitting offset from it.
label_times = hop_times[mask] + window_seconds - label_lag_s
pitch_seq = np.where(note_idx_seq[mask] == SILENCE_INDEX, np.nan,
                     note_idx_seq[mask] + MIDI_LOW - 1)
ax2.plot(label_times, pitch_seq, color=CAT[0], linewidth=1.5, label="derived frame label")
onset_times = (hop_times[mask] + window_seconds)[onset_seq[mask] > 0.5]
ax2.scatter(onset_times, np.full_like(onset_times, MIDI_HIGH + 2), marker="v",
            color=CAT[7], s=18, label="onset frame (plotted at window end)")
ax2.set_xlim(0, t_max)
ax2.set_xlabel("time (s)"); ax2.set_ylabel("class pitch")
ax2.legend(frameon=False, loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

## Step 5 -- Windowing, normalization, caching and augmentation

**Normalization.** Every window is zero-meaned, then divided by `max(peak, NORM_PEAK_FLOOR)`.
This is unconditional -- not "divide by peak only if peak exceeds 1.0" -- because
`librosa.load()` already returns audio in `[-1, 1]`, so a peak-only-if-clipping guard would
never fire on real recordings, and any window quieter than 0 dBFS (i.e. essentially all of
them) would reach the quantizer effectively unnormalized. Dividing unconditionally by
`max(peak, floor)` means any window at or above the floor uses the full int8 range once
quantized, while windows quieter than the floor stay proportionally quiet rather than having
their noise floor amplified to full scale. `normalize_window_np()` and `normalize_window_tf()`
below must stay bit-for-bit equivalent to each other and to `quantize_and_load_input()` in the
firmware -- checked directly in the "normalization parity check" cell further down.

**Cache layout.** The on-disk `tf.data` cache stores **raw** (un-normalized, un-augmented)
windows; normalization and augmentation are applied downstream of the cache. This means tuning
either one doesn't require rebuilding the cache, and augmentation genuinely varies per epoch
rather than being frozen into the cache on the first pass. The cache directory is namespaced
by a fingerprint hash of every parameter that changes `generate_windows()`'s output, with a
`cache_meta.json` sidecar written and verified on load -- `ds.cache(path)` does no content
validation of its own, so this is what turns a stale-cache mismatch into a loud error instead
of a silent one.

In [ ]:
def _rms_dbfs(window: np.ndarray) -> float:
    rms = np.sqrt(np.mean(window.astype(np.float64) ** 2)) + 1e-12
    return 20.0 * math.log10(rms)


def _load_audio(path) -> np.ndarray:
    audio, _ = librosa.load(str(path), sr=SAMPLE_RATE, mono=True)
    return audio.astype(np.float32)


def normalize_window_np(window: np.ndarray) -> np.ndarray:
    """Reference (NumPy) implementation of the window normalization.

    THIS MUST STAY BIT-FOR-BIT EQUIVALENT to normalize_window_tf() below and to
    quantize_and_load_input() in guitar_midi_esp32s3.ino. All three are checked against
    each other in the "normalization parity" cell further down.

    Zero-mean, then divide by max(peak, NORM_PEAK_FLOOR):
      - peak >= floor  ->  output peak is exactly 1.0, using the full int8 input range
      - peak <  floor  ->  output stays below 1.0, so loudness remains a usable cue for
                           the silence class instead of room noise being amplified to
                           full scale
    """
    w = window - np.mean(window)
    peak = float(np.max(np.abs(w)))
    return (w / max(peak, NORM_PEAK_FLOOR)).astype(np.float32)


def normalize_window_tf(x):
    """tf.data implementation of normalize_window_np(). Runs after the cache."""
    x = x - tf.reduce_mean(x)
    peak = tf.reduce_max(tf.abs(x))
    return x / tf.maximum(peak, tf.constant(NORM_PEAK_FLOOR, tf.float32))


def generate_windows(files):
    """Python generator yielding (raw_window[WINDOW,1,1] float32, note_idx, onset_flag) for
    every hop-aligned window across every record in `files`.

    Yields the RAW (un-normalized, DC-included) window on purpose -- normalization lives
    downstream of the cache so that changing it doesn't invalidate the cache and so that
    augmentation is applied to the raw signal before normalization. The RMS silence
    override stays here because it has to see the true signal level.

    Shape is (WINDOW_SAMPLES, 1, 1) so the model can use Conv2D/DepthwiseConv2D directly
    (see the model section for why 2D rather than 1D).
    """
    for rec in files:
        note_idx_seq, onset_seq = frame_labels_by_stem[rec["stem"]]
        audio = _load_audio(rec["wav"])
        if len(audio) < WINDOW_SAMPLES:
            audio = np.pad(audio, (0, WINDOW_SAMPLES - len(audio)))

        # The JAMS-reported duration and the decoded WAV length can differ by a few ms; cap
        # at whichever gives fewer hops so we never index past either array.
        n_hops_audio = 1 + max(0, len(audio) - WINDOW_SAMPLES) // HOP_SAMPLES
        n_hops = min(n_hops_audio, len(note_idx_seq))

        for hop_i in range(n_hops):
            start = hop_i * HOP_SAMPLES
            window = audio[start:start + WINDOW_SAMPLES]
            if len(window) < WINDOW_SAMPLES:
                window = np.pad(window, (0, WINDOW_SAMPLES - len(window)))

            this_note_idx = int(note_idx_seq[hop_i])
            onset = float(onset_seq[hop_i])

            if _rms_dbfs(window) < SILENCE_RMS_DBFS:
                this_note_idx = SILENCE_INDEX
                onset = 0.0

            yield (window.astype(np.float32)[:, np.newaxis, np.newaxis],
                   np.int32(this_note_idx), np.float32(onset))


OUTPUT_SIGNATURE = (
    tf.TensorSpec(shape=(WINDOW_SAMPLES, 1, 1), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.int32),
    tf.TensorSpec(shape=(), dtype=tf.float32),
)

### Augmentation (train split only)

Applied to the **raw** window, before normalization, so that a random gain change actually
interacts with `NORM_PEAK_FLOOR` the way a real level change would rather than being
normalized away into a no-op.

- **Gain** `AUG_GAIN_DB`: the mic distance and the player's dynamics vary; the model should
  not key on absolute level except near the silence floor.
- **Additive white noise** at `AUG_SNR_DB`: the INMP441 in a room is meaningfully noisier
  than a GuitarSet studio reference-mic recording. This is the augmentation that matters
  most for the train-to-deploy gap.
- **Polarity flip**: a mic or pickup can be wired either way round, and a
  polarity-sensitive pitch classifier is memorising something it shouldn't.

Deliberately **not** included: time stretching or pitch shifting. Pitch shifting would
invalidate the note label, and time stretching changes the relationship between the window
length and a note's attack transient, which the onset head depends on.

In [ ]:
def augment_raw_window(x, note, onset):
    """Random gain / noise / polarity on the raw window. Train split only."""
    if AUG_GAIN_DB is not None:
        gain_db = tf.random.uniform([], AUG_GAIN_DB[0], AUG_GAIN_DB[1])
        x = x * tf.pow(10.0, gain_db / 20.0)

    if AUG_SNR_DB is not None:
        snr_db = tf.random.uniform([], AUG_SNR_DB[0], AUG_SNR_DB[1])
        sig_rms = tf.sqrt(tf.reduce_mean(tf.square(x)) + 1e-12)
        noise_rms = sig_rms * tf.pow(10.0, -snr_db / 20.0)
        x = x + tf.random.normal(tf.shape(x), stddev=noise_rms)

    if AUG_POLARITY_FLIP:
        flip = tf.where(tf.random.uniform([]) < 0.5, -1.0, 1.0)
        x = x * flip

    return x, note, onset


def _cache_fingerprint() -> str:
    """Hash of every parameter that changes what generate_windows() produces.

    Used to namespace the on-disk tf.data cache so that changing any of these automatically
    lands in a fresh directory instead of tf.data silently replaying stale windows --
    ds.cache(path) does no content validation of its own.

    Note what is NOT in here: normalization and augmentation parameters. Those are applied
    downstream of the cache now, so changing them doesn't require a rebuild.
    """
    key = repr((
        "v3-raw-cache",
        SAMPLE_RATE, WINDOW_SAMPLES, HOP_SAMPLES, MIDI_LOW, MIDI_HIGH, num_classes,
        ONSET_WINDOW_MS, LABEL_LAG_MS, SILENCE_RMS_DBFS,
        sorted(TEST_PLAYERS), VAL_FRACTION, SEED,
        sorted(r["stem"] for r in records),
    ))
    return hashlib.md5(key.encode()).hexdigest()[:10]


CACHE_DIR = OUT_DIR / "tfdata_cache" / _cache_fingerprint()
CACHE_META = CACHE_DIR / "cache_meta.json"
CACHE_KEY = {
    "sample_rate": SAMPLE_RATE, "window_samples": WINDOW_SAMPLES, "hop_samples": HOP_SAMPLES,
    "midi_low": MIDI_LOW, "midi_high": MIDI_HIGH, "num_classes": num_classes,
    "onset_window_ms": ONSET_WINDOW_MS, "label_lag_ms": LABEL_LAG_MS,
    "silence_rms_dbfs": SILENCE_RMS_DBFS, "test_players": sorted(TEST_PLAYERS),
    "val_fraction": VAL_FRACTION, "seed": SEED, "n_records": len(records),
}

if FORCE_REBUILD_CACHE and CACHE_DIR.exists():
    import shutil
    shutil.rmtree(CACHE_DIR)
    print(f"FORCE_REBUILD_CACHE: removed {CACHE_DIR}")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
if CACHE_META.exists():
    existing = json.loads(CACHE_META.read_text())
    if existing != CACHE_KEY:
        raise RuntimeError(
            f"Cache at {CACHE_DIR} was built with different parameters than the current "
            f"config, despite hashing to the same fingerprint.\nOn disk: {existing}\n"
            f"Now:     {CACHE_KEY}\nDelete that directory (or set FORCE_REBUILD_CACHE=True) "
            "and re-run."
        )
    print(f"Cache metadata verified against the current config: {CACHE_META}")
else:
    CACHE_META.write_text(json.dumps(CACHE_KEY, indent=2))
    print(f"Wrote cache metadata: {CACHE_META}")

print(f"Cache directory: {CACHE_DIR}")


def make_dataset(files, split: str, shuffle_buffer: int = 8192,
                 silence_keep_prob: float = 1.0, class_weights: np.ndarray = None):
    """Builds the windowed tf.data pipeline for `files`.

    Pipeline order matters and is deliberate:

        from_generator (RAW windows)
          -> cache            expensive librosa decode + windowing happens once, ever
          -> [train] filter   silence downsampling
          -> [train] shuffle
          -> [train] augment  random per epoch, because it is AFTER the cache
          -> normalize        peak normalization with floor
          -> [train] weight   per-class sample weights for the note head
          -> batch / prefetch

    Augmenting before the cache would freeze one realization of the randomness into the
    cache file and replay it identically every epoch.
    """
    is_train = split == "train"
    cache_path = str(CACHE_DIR / split)

    ds = tf.data.Dataset.from_generator(lambda: generate_windows(files),
                                        output_signature=OUTPUT_SIGNATURE)
    ds = ds.cache(cache_path)

    if is_train and silence_keep_prob < 1.0:
        keep_p = tf.constant(silence_keep_prob, tf.float32)
        ds = ds.filter(lambda x, note, onset: tf.logical_or(
            tf.not_equal(note, SILENCE_INDEX), tf.random.uniform([]) < keep_p))

    if is_train:
        ds = ds.shuffle(shuffle_buffer, seed=SEED, reshuffle_each_iteration=True)
        if AUGMENT:
            ds = ds.map(augment_raw_window, num_parallel_calls=tf.data.AUTOTUNE)

    if is_train and class_weights is not None:
        cw = tf.constant(class_weights, tf.float32)

        def to_supervised_weighted(x, note, onset):
            return (
                normalize_window_tf(x),
                {"note_output": note, "onset_output": onset},
                {"note_output": tf.gather(cw, note), "onset_output": tf.constant(1.0)},
            )

        ds = ds.map(to_supervised_weighted, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        def to_supervised(x, note, onset):
            return normalize_window_tf(x), {"note_output": note, "onset_output": onset}

        ds = ds.map(to_supervised, num_parallel_calls=tf.data.AUTOTUNE)

    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


def raw_eval_stream(files):
    """(normalized_window, note_idx, onset_flag) triples, un-cached and un-augmented.

    This is the single source of truth for every manual evaluation harness below (the Keras
    direct checkpoint, every TFLite checkpoint, and the representative dataset), so all of
    them see exactly the same windows -- which is what makes the checkpoint ladder's
    comparisons meaningful.
    """
    for x, note, onset in generate_windows(files):
        yield normalize_window_np(x), int(note), float(onset)

### Visualization -- what the model actually sees: one analysis window

A real annotated attack from the training set, shown at three zoom levels: where the analysis
window sits relative to the surrounding audio, the raw vs. normalized window the model
actually receives, and a spectrogram of that window's content. This is the input
representation every architecture decision below is built around.

In [ ]:
_example_rec = train_files[0]
_example_audio = _load_audio(_example_rec["wav"])
_example_events, _ = per_file_events[_example_rec["stem"]]
_onset_time = next((t for (t, end, s, p) in _example_events if t > 1.0), _example_events[0][0])

_win_end_sample = int(round((_onset_time + onset_window_s * 0.5) * SAMPLE_RATE))
_win_start_sample = max(0, _win_end_sample - WINDOW_SAMPLES)
_raw_window = _example_audio[_win_start_sample:_win_start_sample + WINDOW_SAMPLES]
if len(_raw_window) < WINDOW_SAMPLES:
    _raw_window = np.pad(_raw_window, (0, WINDOW_SAMPLES - len(_raw_window)))
_norm_window = normalize_window_np(_raw_window[:, np.newaxis, np.newaxis]).squeeze()

fig, axes = plt.subplots(3, 1, figsize=(11, 8.5))

t_axis_full = np.arange(len(_example_audio)) / SAMPLE_RATE
lo = _win_start_sample / SAMPLE_RATE - 0.05
hi = (_win_start_sample + WINDOW_SAMPLES) / SAMPLE_RATE + 0.05
axes[0].plot(t_axis_full, _example_audio, color=MUTED, linewidth=0.6)
axes[0].axvspan(_win_start_sample / SAMPLE_RATE, (_win_start_sample + WINDOW_SAMPLES) / SAMPLE_RATE,
                 color=SEQ_BLUE[1], alpha=0.6, label=f"{window_seconds * 1000:.0f} ms analysis window")
axes[0].axvline(_onset_time, color=CAT[7], linestyle="--", linewidth=1.2, label="annotated attack")
axes[0].set_xlim(lo, hi)
axes[0].set_ylabel("amplitude")
axes[0].set_title(f"{_example_rec['stem']} -- one analysis window in context")
axes[0].legend(frameon=False, fontsize=8)

t_window_ms = np.arange(WINDOW_SAMPLES) / SAMPLE_RATE * 1000
axes[1].plot(t_window_ms, _raw_window, color=SEQ_BLUE[3], linewidth=0.8, label="raw")
axes[1].plot(t_window_ms, _norm_window, color=CAT[1], linewidth=0.8, label="normalized (model input)")
axes[1].axvspan(window_seconds * 1000 - ONSET_WINDOW_MS, window_seconds * 1000,
                 color=CAT[7], alpha=0.15, label=f"onset band ({ONSET_WINDOW_MS:.0f} ms)")
axes[1].set_xlabel("time within window (ms)"); axes[1].set_ylabel("amplitude")
axes[1].legend(frameon=False, fontsize=8)
axes[1].set_title("Raw vs. normalized window -- exactly what quantize_and_load_input() reproduces on-device")

f_spec, t_spec, Sxx = scipy.signal.spectrogram(_norm_window, fs=SAMPLE_RATE, nperseg=256, noverlap=224)
im = axes[2].pcolormesh(t_spec * 1000, f_spec, 10 * np.log10(Sxx + 1e-12), cmap="Blues", shading="auto")
axes[2].set_ylim(0, 2000)
axes[2].set_xlabel("time within window (ms)"); axes[2].set_ylabel("frequency (Hz)")
axes[2].set_title("Spectrogram of the normalized window (the conv stem's receptive field)")
fig.colorbar(im, ax=axes[2], fraction=0.046, label="dB")

fig.tight_layout()
plt.show()

### Normalization parity check

The single most important invariant in this project is that the notebook and the ESP32
firmware normalize a window *identically*. If they don't, the model is served a different
input distribution on-device than it was trained on, and the resulting accuracy loss looks
exactly like a quantization problem while having nothing to do with quantization.

This cell checks the NumPy reference against the tf.data implementation, and prints the
exact constants the firmware must use (they are also written into `class_labels.h`
automatically at the end of the notebook, and the firmware checks for them at compile time).

In [ ]:
_probe = rng.standard_normal((WINDOW_SAMPLES, 1, 1)).astype(np.float32)
for scale, tag in [(0.9, "loud"), (0.05, "moderate"), (0.001, "below the floor")]:
    probe = _probe * scale + 0.02   # include a DC offset so the centring is exercised too
    a = normalize_window_np(probe)
    b = normalize_window_tf(tf.constant(probe)).numpy()
    max_abs_diff = float(np.max(np.abs(a - b)))
    print(f"{tag:>16s}: raw peak {np.max(np.abs(probe - probe.mean())):.4f} -> "
          f"normalized peak {np.max(np.abs(a)):.4f}   (numpy vs tf max diff {max_abs_diff:.2e})")
    assert max_abs_diff < 1e-5, "NumPy and tf.data normalization disagree"

print("\nOK -- NumPy and tf.data normalization agree.")
print("\nConstants the ESP32 firmware must match (auto-exported to class_labels.h):")
print(f"  WINDOW_SAMPLES        = {WINDOW_SAMPLES}")
print(f"  HOP_SAMPLES           = {HOP_SAMPLES}")
print(f"  SAMPLE_RATE           = {SAMPLE_RATE}")
print(f"  NORM_PEAK_FLOOR       = {NORM_PEAK_FLOOR:.6f}f   ({NORM_PEAK_FLOOR_DBFS:.0f} dBFS)")
print(f"  ONSET_WINDOW_MS       = {ONSET_WINDOW_MS:.0f}")

### Build the datasets

Silence downsampling and the note-head class weights are computed from the *training*
frames only, and applied to the training split only -- validation and test always see the
true class distribution, so the reported numbers stay honest.

In [ ]:
# --- Class statistics from the TRAIN split only ----------------------------
train_note_idx = np.concatenate([frame_labels_by_stem[r["stem"]][0] for r in train_files])
train_counts = np.bincount(train_note_idx, minlength=num_classes).astype(np.float64)
train_silence_fraction = train_counts[SILENCE_INDEX] / train_counts.sum()

# Silence downsampling: keep silence frames with probability p such that silence ends up at
# about MAX_SILENCE_FRACTION of the training stream.
non_silence = train_counts.sum() - train_counts[SILENCE_INDEX]
target_silence = MAX_SILENCE_FRACTION / (1.0 - MAX_SILENCE_FRACTION) * non_silence
silence_keep_prob = float(min(1.0, target_silence / max(1.0, train_counts[SILENCE_INDEX])))
print(f"Train silence fraction before downsampling: {train_silence_fraction:.3f}")
print(f"Silence keep probability: {silence_keep_prob:.3f} "
      f"-> target silence fraction {MAX_SILENCE_FRACTION:.2f}")

# Per-class sample weights, w ~ 1/sqrt(freq), normalized to mean 1 and clipped.
effective_counts = train_counts.copy()
effective_counts[SILENCE_INDEX] *= silence_keep_prob
effective_counts = np.maximum(effective_counts, 1.0)
class_weights = 1.0 / np.sqrt(effective_counts)
class_weights = class_weights / class_weights.mean()
class_weights = np.clip(class_weights, *CLASS_WEIGHT_CLIP).astype(np.float32)
print(f"Class weights: min {class_weights.min():.3f}  max {class_weights.max():.3f}  "
      f"(silence weight {class_weights[SILENCE_INDEX]:.3f})")

# Onset positive rate on the train split, used to balance the focal loss below.
train_onset = np.concatenate([frame_labels_by_stem[r["stem"]][1] for r in train_files])
onset_pos_rate = float(np.mean(train_onset > 0.5))
print(f"Train onset-positive rate: {onset_pos_rate:.4f}")

print("\nBuilding windowed tf.data pipelines "
      "(first pass decodes all audio and fills the cache -- this is the slow one)...")
train_ds = make_dataset(train_files, "train",
                        silence_keep_prob=silence_keep_prob, class_weights=class_weights)
val_ds = make_dataset(val_files, "val")
test_ds = make_dataset(test_files, "test")

## Model: MobileNet-style trunk over raw audio, two position-aware heads

```
  Input [WINDOW_SAMPLES x 1 x 1] raw normalized audio
    -> Conv2D(k=64, f=_w(48), s=STEM_STRIDE) + BN + ReLU
    -> DW-separable block 1 (k=32, f=_w(64))  + MaxPool(2)
    -> DW-separable block 2 (k=16, f=_w(96))  + MaxPool(2)
    -> DW-separable block 3 (k=8,  f=_w(128)) + MaxPool(2)
    -> DW-separable block 4 (k=4,  f=_w(128)) + MaxPool(2)
        |
        +-> note head:  Conv2D(_w(32),1x1) -> MaxPool(4) -> Flatten -> Dense(_w(160)) -> Dense(N, softmax)
        +-> onset head: Conv2D(_w(16),1x1) -> Flatten -> Dense(_w(32)) -> Dense(1, sigmoid)
```

`_w(n)` scales every filter count by `MODEL_WIDTH`, rounded to a multiple of 8 to keep
esp-nn's int8 kernels on their fast paths on the ESP32-S3. The rendered diagram below
(generated directly from the built model's own layer shapes, so it can't drift out of sync
with the code) shows the exact sizes for the current `MODEL_WIDTH` / `STEM_STRIDE`.

**Why `Conv2D` instead of `Conv1D`.** Convolving with a `(k, 1)` kernel over a `(T, 1, C)`
tensor is mathematically identical to a 1D convolution over `(T, C)`, is what the TFLite
converter lowers `Conv1D` to internally regardless, and maps directly onto the `CONV_2D` /
`DEPTHWISE_CONV_2D` kernels esp-nn accelerates on the S3's vector unit. It's also what makes
quantization-aware training available for this trunk: `tfmot`'s 8-bit scheme has no
`QuantizeConfig` registered for `DepthwiseConv1D` (see the checkpoint ladder below).

**Why the heads keep temporal position instead of pooling it away.** A plain global-average
pool over the whole trunk output is translation-invariant by construction, which throws away
*when* within the window something happened. Under the end-referenced label convention (Step
4), both heads need that: the onset head's entire job is to detect energy change near the
window's trailing edge, which a time-average can't represent at all, and the note head needs
to weight the most recent part of the window more heavily than a note that ended 100 ms
earlier. Both heads therefore reduce channels with a cheap 1x1 convolution first and then
flatten a short temporal axis, keeping position information while bounding the parameter
count. This is the single largest architectural contributor to note/onset accuracy.

In [ ]:
def _w(n):
    """Scale a filter count by MODEL_WIDTH, rounded to a multiple of 8 (which keeps
    the esp-nn int8 kernels on their fast paths)."""
    return max(8, int(round(n * MODEL_WIDTH / 8.0)) * 8)


def ds_conv_block(x, kernel_size, filters, pool_size, name):
    """Depthwise-separable block over the time axis of a (T, 1, C) tensor."""
    x = layers.DepthwiseConv2D(kernel_size=(kernel_size, 1), strides=(1, 1), padding="same",
                               depth_multiplier=1, use_bias=False, name=f"{name}_dwconv")(x)
    x = layers.BatchNormalization(name=f"{name}_dwbn")(x)
    x = layers.ReLU(name=f"{name}_dwrelu")(x)
    x = layers.Conv2D(filters=filters, kernel_size=(1, 1), strides=(1, 1), padding="same",
                      use_bias=False, name=f"{name}_pwconv")(x)
    x = layers.BatchNormalization(name=f"{name}_pwbn")(x)
    x = layers.ReLU(name=f"{name}_pwrelu")(x)
    if pool_size > 1:
        x = layers.MaxPooling2D(pool_size=(pool_size, 1), name=f"{name}_pool")(x)
    return x


def build_model(num_classes: int) -> keras.Model:
    inputs = layers.Input(shape=(WINDOW_SAMPLES, 1, 1), name="audio_input")

    # Stem: a wide receptive field over raw PCM, strided so the trunk downstream sees
    # WINDOW_SAMPLES / STEM_STRIDE timesteps regardless of the window length.
    x = layers.Conv2D(_w(48), kernel_size=(64, 1), strides=(STEM_STRIDE, 1), padding="same",
                      use_bias=False, name="stem_conv")(inputs)
    x = layers.BatchNormalization(name="stem_bn")(x)
    x = layers.ReLU(name="stem_relu")(x)

    x = ds_conv_block(x, kernel_size=32, filters=_w(64),  pool_size=2, name="ds1")
    x = ds_conv_block(x, kernel_size=16, filters=_w(96),  pool_size=2, name="ds2")
    x = ds_conv_block(x, kernel_size=8,  filters=_w(128), pool_size=2, name="ds3")
    trunk = ds_conv_block(x, kernel_size=4, filters=_w(128), pool_size=2, name="ds4")

    # --- Note head: position-aware, but pooled down hard to bound the parameter count ---
    n = layers.Conv2D(_w(32), (1, 1), padding="same", use_bias=False, name="note_reduce")(trunk)
    n = layers.BatchNormalization(name="note_reduce_bn")(n)
    n = layers.ReLU(name="note_reduce_relu")(n)
    n = layers.MaxPooling2D(pool_size=(4, 1), name="note_pool")(n)
    n = layers.Flatten(name="note_flatten")(n)
    n = layers.Dense(_w(160), activation="relu", name="note_dense")(n)
    n = layers.Dropout(DROPOUT, name="note_dropout")(n)
    note_output = layers.Dense(num_classes, activation="softmax", name="note_output")(n)

    # --- Onset head: full temporal resolution of the trunk, few channels ---
    o = layers.Conv2D(_w(16), (1, 1), padding="same", use_bias=False, name="onset_reduce")(trunk)
    o = layers.BatchNormalization(name="onset_reduce_bn")(o)
    o = layers.ReLU(name="onset_reduce_relu")(o)
    o = layers.Flatten(name="onset_flatten")(o)
    o = layers.Dense(_w(32), activation="relu", name="onset_dense")(o)
    onset_output = layers.Dense(1, activation="sigmoid", name="onset_output")(o)

    return models.Model(inputs=inputs, outputs=[note_output, onset_output],
                        name="guitar_midi_dscnn")


model = build_model(num_classes)
model.summary()

total_params = model.count_params()
print(f"\nTotal parameters: {total_params:,}")
print(f"  ~{total_params * 4 / 1024:.1f} KB as FP32")
print(f"  ~{total_params / 1024:.1f} KB after INT8 quantization (150 KB budget in the spec)")


def estimate_macs(model) -> int:
    """Rough multiply-accumulate count per inference."""
    total = 0
    for layer in model.layers:
        cfg = layer.get_config()
        if isinstance(layer, layers.Conv2D) and not isinstance(layer, layers.DepthwiseConv2D):
            out = layer.output_shape
            k = cfg["kernel_size"]
            total += out[1] * out[2] * out[3] * k[0] * k[1] * layer.input_shape[-1]
        elif isinstance(layer, layers.DepthwiseConv2D):
            out = layer.output_shape
            k = cfg["kernel_size"]
            total += out[1] * out[2] * out[3] * k[0] * k[1]
        elif isinstance(layer, layers.Dense):
            total += layer.input_shape[-1] * cfg["units"]
    return total


macs = estimate_macs(model)
TARGET_INFERENCE_PERIOD_MS = 20  # must match TARGET_INFERENCE_PERIOD_MS in guitar_midi_esp32s3.ino
print(f"\nEstimated MACs per window: {macs / 1e6:.2f} M")
print("The firmware runs inference on every 10 ms hop (no decimation) and measures its own")
print(f"Invoke() time against a {TARGET_INFERENCE_PERIOD_MS} ms real-time target, reporting the result in its")
print("periodic [HEALTH] serial line. See the capacity-vs-budget chart below for how")
print("STEM_STRIDE and MODEL_WIDTH were chosen to fit inside that budget.")

### Visualization -- model architecture

In [ ]:
def draw_model_architecture(model):
    """Renders a schematic of build_model()'s graph directly from the built model's own
    layer shapes, so this figure can never drift out of sync with the code that produced it.
    """
    def out_shape(name):
        return model.get_layer(name).output_shape

    def shape_label(shp):
        dims = [d for d in shp[1:] if d != 1]
        return "x".join(str(d) for d in dims) if dims else str(shp[-1])

    trunk_stages = [
        ("input", "audio_input", "raw audio"),
        ("stem", "stem_relu", f"Conv2D k=64 s={STEM_STRIDE}"),
        ("ds1", "ds1_pool", "DW-sep k=32"),
        ("ds2", "ds2_pool", "DW-sep k=16"),
        ("ds3", "ds3_pool", "DW-sep k=8"),
        ("ds4", "ds4_pool", "DW-sep k=4"),
    ]
    note_stages = [
        ("note_reduce", "note_reduce_relu", "Conv2D 1x1"),
        ("note_pool", "note_pool", "MaxPool(4)"),
        ("note_dense", "note_dense", "Dense+ReLU"),
        ("note_output", "note_output", "Dense+softmax"),
    ]
    onset_stages = [
        ("onset_reduce", "onset_reduce_relu", "Conv2D 1x1"),
        ("onset_dense", "onset_dense", "Dense+ReLU"),
        ("onset_output", "onset_output", "Dense+sigmoid"),
    ]

    fig, ax = plt.subplots(figsize=(15, 6))
    box_w, box_h, gap = 1.7, 1.0, 0.5
    trunk_y = 1.6

    def draw_box(x, y, label, sublabel, color):
        ax.add_patch(plt.Rectangle((x, y), box_w, box_h, facecolor=color,
                                    edgecolor=INK, linewidth=1.0, zorder=2))
        ax.text(x + box_w / 2, y + box_h * 0.65, label, ha="center", va="center",
                fontsize=9, color=INK, zorder=3)
        ax.text(x + box_w / 2, y + box_h * 0.28, sublabel, ha="center", va="center",
                fontsize=7.5, color=INK2, zorder=3)

    x = 0.0
    for i, (short, layer_name, op_label) in enumerate(trunk_stages):
        shp = out_shape(layer_name)
        draw_box(x, trunk_y, short, f"{op_label}\n{shape_label(shp)}",
                 SEQ_BLUE[min(i, len(SEQ_BLUE) - 1)])
        if i < len(trunk_stages) - 1:
            ax.annotate("", xy=(x + box_w + gap, trunk_y + box_h / 2),
                        xytext=(x + box_w, trunk_y + box_h / 2),
                        arrowprops=dict(arrowstyle="->", color=MUTED))
        x += box_w + gap

    split_x = x
    head_y_note = trunk_y + 1.6
    head_y_onset = trunk_y - 1.6
    ax.annotate("", xy=(split_x, head_y_note + box_h / 2), xytext=(split_x - gap, trunk_y + box_h / 2),
                arrowprops=dict(arrowstyle="->", color=CAT[0]))
    ax.annotate("", xy=(split_x, head_y_onset + box_h / 2), xytext=(split_x - gap, trunk_y + box_h / 2),
                arrowprops=dict(arrowstyle="->", color=CAT[1]))

    hx = split_x
    for stages, y, color, tag in [(note_stages, head_y_note, CAT[0], "note head"),
                                   (onset_stages, head_y_onset, CAT[1], "onset head")]:
        hx = split_x
        for i, (short, layer_name, op_label) in enumerate(stages):
            shp = out_shape(layer_name)
            draw_box(hx, y, short, f"{op_label}\n{shape_label(shp)}", color)
            if i < len(stages) - 1:
                ax.annotate("", xy=(hx + box_w + gap, y + box_h / 2), xytext=(hx + box_w, y + box_h / 2),
                            arrowprops=dict(arrowstyle="->", color=MUTED))
            hx += box_w + gap
        ax.text(hx + 0.1, y + box_h / 2, tag, va="center", fontsize=10, color=color, weight="bold")

    ax.set_xlim(-0.5, hx + 2.5)
    ax.set_ylim(head_y_onset - 0.8, head_y_note + box_h + 0.8)
    ax.axis("off")
    ax.set_title(f"guitar_midi_dscnn -- {model.count_params():,} params, "
                 f"{estimate_macs(model) / 1e6:.2f} MMAC/window", fontsize=12)
    fig.tight_layout()
    plt.show()


draw_model_architecture(model)

### Visualization -- model capacity vs. the firmware's real-time inference budget

The firmware must finish one `Invoke()` call comfortably inside its target inference period
(`TARGET_INFERENCE_PERIOD_MS` in `guitar_midi_esp32s3.ino`) for onset timing to stay accurate
-- a model that runs slower than the onset band it was trained to detect will sample straight
past the attack. This chart sweeps `STEM_STRIDE` / `MODEL_WIDTH` combinations, builds each one
(without training, purely to count MACs via `estimate_macs`), and converts MACs to predicted
inference time using the measured on-device throughput of the target ESP32-S3 board
(~91.4 MMAC/s on esp-nn's INT8 kernels). The configuration actually used by this notebook
(from the CONFIG cell) is highlighted.

In [ ]:
MEASURED_MMAC_PER_SEC = 91.4  # measured Invoke() throughput on the ESP32-S3 target board (esp-nn INT8 kernels)

_candidate_configs = [(4, 1.0), (4, 0.5), (8, 1.0), (8, 0.75), (8, 0.5), (8, 0.35)]
_capacity_rows = []
_orig_width, _orig_stride = MODEL_WIDTH, STEM_STRIDE
for stride, width in _candidate_configs:
    globals()["MODEL_WIDTH"], globals()["STEM_STRIDE"] = width, stride
    probe_model = build_model(num_classes)
    probe_macs = estimate_macs(probe_model)
    _capacity_rows.append({
        "label": f"stride {stride}\nwidth {width}",
        "stride": stride, "width": width,
        "mmac": probe_macs / 1e6,
        "predicted_ms": (probe_macs / 1e6) / MEASURED_MMAC_PER_SEC * 1000.0,
        "chosen": (stride == _orig_stride and width == _orig_width),
    })
globals()["MODEL_WIDTH"], globals()["STEM_STRIDE"] = _orig_width, _orig_stride

cap_df = pd.DataFrame(_capacity_rows)
print(cap_df[["label", "mmac", "predicted_ms", "chosen"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4.5))
colors = [CAT[1] if r["chosen"] else SEQ_BLUE[3] for r in _capacity_rows]
bars = ax.bar([r["label"] for r in _capacity_rows], cap_df["predicted_ms"], color=colors)
ax.axhline(TARGET_INFERENCE_PERIOD_MS, color=CAT[7], linestyle="--", linewidth=1.5,
           label=f"{TARGET_INFERENCE_PERIOD_MS} ms real-time budget")
for bar, r in zip(bars, _capacity_rows):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{r['mmac']:.2f} MMAC", ha="center", fontsize=8, color=INK2)
ax.set_ylabel("predicted Invoke() time (ms)")
ax.set_title("Model capacity vs. real-time inference budget on the ESP32-S3\n"
             "(highlighted bar = the configuration this notebook is currently set to)")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## Combined loss

```
Loss_total = Loss_NoteClassification + lambda * Loss_OnsetFocalBCE
```

**Why the onset loss is focal rather than plain cross-entropy.** Onset-positive frames are a
small percentage of the corpus (see Step 4). Under plain `BinaryCrossentropy`, the cheapest
thing the network can do is predict "no onset" everywhere, and under `BinaryAccuracy` that
strategy scores extremely well despite being useless -- a constant-zero predictor already
clears the high-90s. Two choices follow from that:

- **Loss**: `BinaryFocalCrossentropy(apply_class_balancing=True)` down-weights the easy
  negatives and up-weights the rare positives, with `alpha` set from the measured positive
  rate.
- **Metrics**: precision, recall and AUC-PR, with F1 computed from precision/recall in the
  evaluation harness. Accuracy is reported for completeness but should not be used to judge
  the onset head.

The note head uses sparse categorical cross-entropy with per-class sample weights supplied by
the dataset, and reports top-1 plus top-3 accuracy -- if top-1 is poor but top-3 is high, the
model has the right pitch region and is confusing neighbouring semitones, which points at
window length rather than model capacity.

In [ ]:
# alpha for class-balanced focal loss: weight on the positive class. Setting it to
# 1 - positive_rate is the usual balanced choice.
ONSET_ALPHA = float(np.clip(1.0 - onset_pos_rate, 0.5, 0.99))
print(f"Onset focal loss: alpha={ONSET_ALPHA:.4f} (from a {onset_pos_rate:.4f} positive rate), gamma=2.0")


def compile_model(m, learning_rate):
    m.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss={
            "note_output": keras.losses.SparseCategoricalCrossentropy(),
            "onset_output": keras.losses.BinaryFocalCrossentropy(
                apply_class_balancing=True, alpha=ONSET_ALPHA, gamma=2.0),
        },
        loss_weights={"note_output": 1.0, "onset_output": LAMBDA_ONSET},
        metrics={
            "note_output": [
                keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
                keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3"),
            ],
            "onset_output": [
                keras.metrics.BinaryAccuracy(name="accuracy"),
                keras.metrics.Precision(name="precision"),
                keras.metrics.Recall(name="recall"),
                keras.metrics.AUC(curve="PR", name="aucpr"),
            ],
        },
    )
    return m


compile_model(model, LEARNING_RATE)

callbacks = [
    # Monitor the note head's val accuracy rather than val_loss: the combined loss is
    # dominated by the onset focal term early in training, so val_loss can still be
    # improving while the thing we actually care about has plateaued (and vice versa).
    keras.callbacks.EarlyStopping(monitor="val_note_output_accuracy", mode="max",
                                  patience=10, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_note_output_accuracy", mode="max",
                                      factor=0.5, patience=4, min_lr=1e-5, verbose=1),
    keras.callbacks.ModelCheckpoint(str(OUT_DIR / "best_by_val_note_acc.keras"),
                                    monitor="val_note_output_accuracy", mode="max",
                                    save_best_only=True, verbose=0),
]

print("\nTraining...")
t0 = time.time()
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
print(f"Training finished in {(time.time() - t0) / 60:.1f} min")

model.save(str(OUT_DIR / "guitar_midi_model.keras"))
print(f"Saved: {OUT_DIR / 'guitar_midi_model.keras'}")

### Training curves

In [ ]:
h = history.history
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(h["loss"], color=CAT[0], label="train")
axes[0].plot(h["val_loss"], color=CAT[1], label="val")
axes[0].set_title("Combined loss"); axes[0].set_xlabel("epoch"); axes[0].legend(frameon=False)

axes[1].plot(h["note_output_accuracy"], color=CAT[0], label="train top-1")
axes[1].plot(h["val_note_output_accuracy"], color=CAT[1], label="val top-1")
if "val_note_output_top3" in h:
    axes[1].plot(h["val_note_output_top3"], color=CAT[1], linestyle="--", label="val top-3")
axes[1].axhline(0.90, color=MUTED, linestyle=":", linewidth=1)
axes[1].set_title("Note head accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend(frameon=False)

if "val_onset_output_precision" in h:
    axes[2].plot(h["val_onset_output_precision"], color=CAT[0], label="val precision")
    axes[2].plot(h["val_onset_output_recall"], color=CAT[1], label="val recall")
    axes[2].plot(h["val_onset_output_aucpr"], color=CAT[2], label="val AUC-PR")
axes[2].set_title("Onset head (accuracy is NOT a useful metric here)")
axes[2].set_xlabel("epoch"); axes[2].legend(frameon=False)

fig.tight_layout()
plt.show()

## Evaluation

In [ ]:
eval_results = model.evaluate(test_ds, return_dict=True)
print("Test set evaluation (Keras, via test_ds):")
for k, v in eval_results.items():
    print(f"  {k}: {v:.4f}")

keras_note_acc = eval_results["note_output_accuracy"]
keras_onset_acc = eval_results["onset_output_accuracy"]

## Quantization robustness checkpoints

Post-training int8 quantization can silently cost real accuracy if the input distribution,
output-tensor identification, or evaluation coverage aren't handled carefully. Rather than
trust a single before/after number, this pipeline evaluates the model at each stage of the
conversion process, so any accuracy loss is isolated to a specific, identifiable cause instead
of attributed to "quantization" as a whole:

| Checkpoint | What it isolates |
|---|---|
| **0a** Keras direct, manual harness | Is `test_ds` (and its cache) trustworthy? No TFLite involved. |
| **0** float32 `.tflite` | Does TFLite *conversion* alone cost anything? No quantization involved. |
| **1** dynamic range (int8 weights, float activations) | Does quantizing the **weights** cost anything? No representative dataset involved. |
| **2** full-integer INT8 | Does quantizing the **activations** cost anything? This is post-training quantization (PTQ). |
| **2b** per-layer debugger | Only runs if 2 is still far below 0/1: which op is responsible? |
| **3** quantization-aware training | **The deployment path.** |

**Checkpoint 3 (QAT) is the deployment path.** Post-training quantization rounds a network
that was never asked to tolerate int8 arithmetic, so whatever accuracy it costs can only be
worked around after the fact via calibration. QAT inserts fake-quantization into the forward
pass during a short fine-tune, so the weights themselves move to values where int8 rounding
doesn't hurt in the first place. This is available here specifically because the model's
trunk is built from `Conv2D`/`DepthwiseConv2D` rather than their 1D equivalents.

### Shared metrics

- **Note accuracy is reported twice**: overall (every frame) and restricted to frames whose
  true label is not silence. Overall accuracy is inflated by the silence class, which is
  both the most common label and the easiest one. The non-silence number is the one that
  actually reflects pitch discrimination.
- **Onset is reported as precision / recall / F1**, not accuracy. At a few percent positive
  rate, "accuracy" rewards a head that never fires.

In [ ]:
def score_predictions(true_notes, pred_notes, true_onset, pred_onset_prob, onset_threshold=0.5):
    """Turns raw per-window predictions into the metric dict used everywhere below."""
    true_notes = np.asarray(true_notes)
    pred_notes = np.asarray(pred_notes)
    true_onset = np.asarray(true_onset) > 0.5
    pred_onset = np.asarray(pred_onset_prob) > onset_threshold
    n = len(true_notes)

    note_correct = pred_notes == true_notes
    nonsilence = true_notes != SILENCE_INDEX

    tp = int(np.sum(pred_onset & true_onset))
    fp = int(np.sum(pred_onset & ~true_onset))
    fn = int(np.sum(~pred_onset & true_onset))
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-9, precision + recall)

    return {
        "n": n,
        "note_acc": float(np.mean(note_correct)),
        "note_acc_nonsilence": float(np.mean(note_correct[nonsilence])) if nonsilence.any() else float("nan"),
        "onset_acc": float(np.mean(pred_onset == true_onset)),
        "onset_precision": precision,
        "onset_recall": recall,
        "onset_f1": f1,
        "onset_threshold": onset_threshold,
        "onset_pos_fraction": float(np.mean(true_onset)),
    }


def print_scores(tag, s):
    print(f"{tag}")
    print(f"    note accuracy        : {s['note_acc'] * 100:6.2f}%   (all frames, n={s['n']:,})")
    print(f"    note acc, non-silence: {s['note_acc_nonsilence'] * 100:6.2f}%   <- the pitch-discrimination number")
    print(f"    onset P / R / F1     : {s['onset_precision'] * 100:5.1f}% / "
          f"{s['onset_recall'] * 100:5.1f}% / {s['onset_f1'] * 100:5.1f}%   "
          f"(threshold {s['onset_threshold']:.2f})")
    print(f"    onset accuracy       : {s['onset_acc'] * 100:6.2f}%   (reported for continuity only; a")
    print(f"                                       constant-'no' predictor scores "
          f"{100 * (1 - s['onset_pos_fraction']):.2f}%)")


results = {}       # checkpoint key -> metric dict
raw_preds = {}     # checkpoint key -> (true_notes, pred_notes, true_onset, pred_onset_prob)

In [ ]:
def collect_tflite_predictions(tflite_path, files, max_windows=None, seed=SEED):
    """Runs any .tflite model (float32 or full-int8 in/out) over `files` and returns raw
    per-window predictions.

    Auto-detects the input dtype, so the same function serves every checkpoint: float32
    models get normalized windows fed straight in, int8 models get quantized first using the
    model's own input scale/zero_point -- exactly what quantize_and_load_input() does
    on-device in guitar_midi_esp32s3.ino.

    argmax(note_out) is taken on the raw tensor (int8 or float) without dequantizing first.
    That's valid because int8 dequantization is a monotonic affine map (scale > 0), so
    argmax over quantized values equals argmax over dequantized values.

    OUTPUT SELECTION IS BY SHAPE, NOT NAME. from_keras_model() conversion routinely discards
    the Keras "note_output"/"onset_output" names -- the interpreter hands back opaque names
    like "StatefulPartitionedCall_1:0", and TFLite's output order in the flatbuffer is not
    guaranteed to match Keras' model.outputs order. The note head has num_classes elements
    and the onset head has exactly 1, which disambiguates them regardless of naming or order.

    max_windows=None (default) evaluates EVERY window, matching model.evaluate(). An explicit
    cap draws a RANDOM subset across all files rather than a sequential prefix, so onset-
    positive frames (which cluster at real note attacks rather than spreading evenly) are
    represented proportionally in any sub-sampled evaluation.
    """
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()

    def last_dim(d):
        return int(d["shape"][-1])

    by_last_dim = {last_dim(d): d for d in output_details}
    if 1 not in by_last_dim or num_classes not in by_last_dim:
        raise RuntimeError(
            "Could not identify note/onset outputs by shape. output_details="
            f"{[(d['name'], d['shape']) for d in output_details]}"
        )
    note_detail = by_last_dim[num_classes]
    onset_detail = by_last_dim[1]

    in_dtype = input_details["dtype"]
    is_int8_io = in_dtype == np.int8
    if is_int8_io:
        in_scale, in_zero_point = input_details["quantization"]
        onset_scale, onset_zero_point = onset_detail["quantization"]

    if max_windows is None:
        window_iter = raw_eval_stream(files)
    else:
        rng_local = np.random.default_rng(seed)
        all_windows = list(raw_eval_stream(files))
        if len(all_windows) > max_windows:
            idx = rng_local.choice(len(all_windows), size=max_windows, replace=False)
            all_windows = [all_windows[i] for i in idx]
        window_iter = all_windows

    true_notes, pred_notes, true_onset, pred_onset_prob = [], [], [], []
    t0 = time.time()
    for x, note_idx, onset_flag in window_iter:
        if is_int8_io:
            x_in = np.clip(np.round(x / in_scale + in_zero_point), -128, 127).astype(in_dtype)
        else:
            x_in = x.astype(in_dtype)

        interpreter.set_tensor(input_details["index"], np.expand_dims(x_in, 0))
        interpreter.invoke()

        note_out = interpreter.get_tensor(note_detail["index"])[0]
        onset_out = interpreter.get_tensor(onset_detail["index"])[0]

        pred_notes.append(int(np.argmax(note_out)))
        if is_int8_io:
            pred_onset_prob.append((float(onset_out.flatten()[0]) - onset_zero_point) * onset_scale)
        else:
            pred_onset_prob.append(float(onset_out.flatten()[0]))
        true_notes.append(int(note_idx))
        true_onset.append(float(onset_flag))
        if len(true_notes) % 10000 == 0:
            print(f"  ...{len(true_notes):,} windows ({time.time() - t0:.0f}s)")

    return (np.array(true_notes), np.array(pred_notes),
            np.array(true_onset), np.array(pred_onset_prob))


def collect_keras_predictions(keras_model, files, batch_size=512):
    """Same as collect_tflite_predictions() but for a raw Keras model -- no TFLite, no
    SavedModel export, no test_ds. Batched for speed; eager per-window calls would be far
    slower than the TFLite interpreter's C++ loop.

    This is what isolates "is test_ds trustworthy?" from "does TFLite conversion introduce
    error?". If this matches model.evaluate(test_ds), test_ds is fine.
    """
    true_notes, pred_notes, true_onset, pred_onset_prob = [], [], [], []
    batch_x, batch_note, batch_onset = [], [], []

    def flush():
        if not batch_x:
            return
        note_out, onset_out = keras_model(np.stack(batch_x, axis=0), training=False)
        pred_notes.extend(np.argmax(note_out.numpy(), axis=1).tolist())
        pred_onset_prob.extend(onset_out.numpy().reshape(-1).tolist())
        true_notes.extend(batch_note)
        true_onset.extend(batch_onset)
        batch_x.clear(); batch_note.clear(); batch_onset.clear()

    for x, note_idx, onset_flag in raw_eval_stream(files):
        batch_x.append(x)
        batch_note.append(int(note_idx))
        batch_onset.append(float(onset_flag))
        if len(batch_x) >= batch_size:
            flush()
    flush()
    return (np.array(true_notes), np.array(pred_notes),
            np.array(true_onset), np.array(pred_onset_prob))

### Checkpoint 0a -- Keras model evaluated directly (no TFLite at all)

Isolates `test_ds` and its on-disk cache from TFLite conversion entirely: this calls the
trained Keras model over the exact same `generate_windows()` source and argmax/threshold
logic every other checkpoint uses, with zero TFLite involved.

Compare against the `model.evaluate(test_ds)` baseline printed just above. **A match confirms
the cache is healthy** -- the fingerprinted, metadata-verified cache directory (Step 5) is
serving exactly what `generate_windows()` produces today. A mismatch means `test_ds` is
serving something else, and nothing below this line can be trusted until that's resolved.

In [ ]:
print("=== Checkpoint 0a: Keras model, direct (no TFLite), manual per-window harness ===")
print(f"Keras model.evaluate(test_ds) baseline: note {keras_note_acc * 100:.2f}%, "
      f"onset acc {keras_onset_acc * 100:.2f}%\n")

raw_preds["0a_keras_direct"] = collect_keras_predictions(model, test_files)
results["0a_keras_direct"] = score_predictions(*raw_preds["0a_keras_direct"])
print_scores("Checkpoint 0a -- Keras direct", results["0a_keras_direct"])

gap = abs(results["0a_keras_direct"]["note_acc"] - keras_note_acc)
print(f"\n  Note-accuracy gap vs. model.evaluate(test_ds): {gap * 100:.2f} points")
if gap > 0.02:
    print("  *** WARNING: a real gap. test_ds is NOT serving the same windows as")
    print("      generate_windows(). Delete artifacts/tfdata_cache/ entirely, set")
    print("      FORCE_REBUILD_CACHE=True, and re-run before trusting anything below.")
else:
    print("  OK -- test_ds and the manual harness agree; the cache is healthy and every")
    print("     checkpoint below is measuring what it claims to measure.")

### Onset threshold selection (on validation, never on test)

The firmware compares the onset head's dequantized sigmoid output against a fixed threshold
(`MODEL_ONSET_THRESHOLD` in `class_labels.h`). 0.5 is rarely the F1-optimal threshold for a
head trained on a heavily imbalanced target, so it is swept on the **validation** split here
and exported for the firmware, keeping the test split clean for reporting.

In [ ]:
val_true_notes, val_pred_notes, val_true_onset, val_pred_prob = collect_keras_predictions(model, val_files)

thresholds = np.linspace(0.05, 0.95, 91)
f1s = []
for t in thresholds:
    s = score_predictions(val_true_notes, val_pred_notes, val_true_onset, val_pred_prob, onset_threshold=float(t))
    f1s.append(s["onset_f1"])
f1s = np.array(f1s)
ONSET_THRESHOLD = float(thresholds[int(np.argmax(f1s))])
print(f"Best onset threshold on VALIDATION: {ONSET_THRESHOLD:.2f}  (F1 {f1s.max() * 100:.1f}%)")
print(f"For reference, an unweighted midpoint threshold of 0.50: F1 "
      f"{f1s[int(np.argmin(np.abs(thresholds - 0.50)))] * 100:.1f}%")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(thresholds, f1s * 100, color=CAT[0])
ax.axvline(ONSET_THRESHOLD, color=CAT[1], linestyle="--", label=f"selected {ONSET_THRESHOLD:.2f}")
ax.axvline(0.50, color=MUTED, linestyle=":", label="unweighted midpoint (0.50)")
ax.set_xlabel("onset probability threshold"); ax.set_ylabel("onset F1 (%)")
ax.set_title("Onset threshold sweep on the validation split")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

# Re-score checkpoint 0a at the selected threshold now that we have one.
results["0a_keras_direct"] = score_predictions(*raw_preds["0a_keras_direct"], onset_threshold=ONSET_THRESHOLD)
print_scores("Checkpoint 0a -- re-scored at the selected threshold", results["0a_keras_direct"])

### Checkpoint 0 -- float32 `.tflite`, no quantization

In [ ]:
print("=== Checkpoint 0: float32 TFLite export (no quantization) ===")
print("Isolates whether TFLite CONVERSION alone costs anything. Should land within noise of")
print("checkpoint 0a -- it is the same weights, just re-exported.\n")

converter_f32 = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_f32 = converter_f32.convert()
f32_path = OUT_DIR / "checkpoint0_float32.tflite"
f32_path.write_bytes(tflite_f32)
print(f"Saved: {f32_path} ({len(tflite_f32) / 1024:.1f} KB)")

# Diagnostic: dump the raw output tensor details so the shape-based disambiguation used by
# collect_tflite_predictions() is visible directly.
_diag = tf.lite.Interpreter(model_path=str(f32_path))
_diag.allocate_tensors()
print("Raw output tensor details (this is what has to be disambiguated by shape):")
for d in _diag.get_output_details():
    print(f"  index={d['index']}  name={d['name']!r}  shape={d['shape']}")
del _diag

raw_preds["0_float32"] = collect_tflite_predictions(f32_path, test_files)
results["0_float32"] = score_predictions(*raw_preds["0_float32"], onset_threshold=ONSET_THRESHOLD)
print_scores("Checkpoint 0 -- float32 TFLite", results["0_float32"])

### Checkpoint 1 -- dynamic-range quantization (int8 weights, float32 activations)

In [ ]:
print("=== Checkpoint 1: dynamic-range quantization (int8 weights, float32 activations) ===")
print("No representative dataset is involved at all, so this isolates whether quantizing the")
print("WEIGHTS costs anything, separately from activation calibration.\n")

converter_dr = tf.lite.TFLiteConverter.from_keras_model(model)
converter_dr.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_dr = converter_dr.convert()
dr_path = OUT_DIR / "checkpoint1_dynamic_range.tflite"
dr_path.write_bytes(tflite_dr)
print(f"Saved: {dr_path} ({len(tflite_dr) / 1024:.1f} KB)")

raw_preds["1_dynamic_range"] = collect_tflite_predictions(dr_path, test_files)
results["1_dynamic_range"] = score_predictions(*raw_preds["1_dynamic_range"], onset_threshold=ONSET_THRESHOLD)
print_scores("Checkpoint 1 -- dynamic range", results["1_dynamic_range"])

### Checkpoint 2 -- full-integer INT8 (post-training quantization)

The representative dataset used for activation calibration is drawn uniformly at random
across the whole training set (reservoir sampling, so the full training set never has to sit
in memory at once). Because every window is peak-normalized to (at most) 1.0 before this
point, the calibration set's input scale lands at approximately `1/127` regardless of which
windows happen to be sampled -- the cell below asserts that directly, so a regression in
normalization anywhere upstream is caught here rather than silently degrading accuracy.

In [ ]:
print("=== Checkpoint 2: full-integer INT8 (weights + activations), PTQ ===\n")

# Materialize the calibration pool once so both this converter and the QAT converter below
# can reuse it without re-decoding all the training audio twice.
print("Materializing the representative-dataset pool...")
_calib_rng = np.random.default_rng(SEED)
_calib_pool = []
_reservoir_target = 4000
for i, (x, _, _) in enumerate(raw_eval_stream(train_files)):
    # Reservoir sampling: a uniform random sample across ALL training windows without
    # holding the entire (many GB) training set in memory at once.
    if len(_calib_pool) < _reservoir_target:
        _calib_pool.append(x)
    else:
        j = _calib_rng.integers(0, i + 1)
        if j < _reservoir_target:
            _calib_pool[int(j)] = x
print(f"Calibration pool: {len(_calib_pool)} windows sampled uniformly from {i + 1:,} training windows")

_calib_peaks = np.array([float(np.max(np.abs(w))) for w in _calib_pool])
print(f"Calibration window peaks: min {_calib_peaks.min():.4f}  median {np.median(_calib_peaks):.4f}  "
      f"max {_calib_peaks.max():.4f}")
print(f"  ({100 * np.mean(_calib_peaks > 0.99):.1f}% sit at exactly 1.0, i.e. above the normalization")
print("   floor -- expected, since normalize_window_np() scales every such window to unit peak,")
print("   so the input tensor's quantization scale uses the full int8 range regardless of which")
print("   windows are sampled for calibration.)")


def representative_data_gen(n_samples=500, seed=SEED):
    rng_local = np.random.default_rng(seed)
    idx = rng_local.choice(len(_calib_pool), size=min(n_samples, len(_calib_pool)), replace=False)
    for i in idx:
        yield [np.expand_dims(_calib_pool[int(i)], axis=0).astype(np.float32)]


def convert_int8(source_model, rep_gen):
    conv = tf.lite.TFLiteConverter.from_keras_model(source_model)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = rep_gen
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8
    conv.inference_output_type = tf.int8
    return conv


converter = convert_int8(model, representative_data_gen)
tflite_int8 = converter.convert()
int8_path = OUT_DIR / "checkpoint2_full_int8.tflite"
int8_path.write_bytes(tflite_int8)
print(f"\nSaved: {int8_path} ({len(tflite_int8) / 1024:.1f} KB)")

# --- Assert the input scale actually uses the full int8 range --------------
_chk = tf.lite.Interpreter(model_path=str(int8_path))
_chk.allocate_tensors()
_in = _chk.get_input_details()[0]
_in_scale, _in_zp = _in["quantization"]
_effective_levels = 2.0 / _in_scale
print(f"INT8 input tensor: scale={_in_scale:.6f}  zero_point={_in_zp}")
print(f"  -> a full-scale [-1, 1] window spans {_effective_levels:.0f} of the 255 available int8 levels")
if _effective_levels < 200:
    print("  *** WARNING: the input quantization is NOT using the full int8 range. Check that")
    print("      normalize_window_np() is actually being applied in raw_eval_stream().")
else:
    print("  OK -- the input quantization uses essentially the whole int8 range.")
del _chk

raw_preds["2_full_int8"] = collect_tflite_predictions(int8_path, test_files)
results["2_full_int8"] = score_predictions(*raw_preds["2_full_int8"], onset_threshold=ONSET_THRESHOLD)
print_scores("Checkpoint 2 -- full INT8 (PTQ)", results["2_full_int8"])

print(f"\nPTQ cost vs. float32 (checkpoint 0): "
      f"{(results['0_float32']['note_acc'] - results['2_full_int8']['note_acc']) * 100:+.2f} points of note accuracy")

### Checkpoint 2b -- per-layer quantization-error debugging (conditional)

Only runs if checkpoint 2 is still meaningfully below checkpoints 0/1, in which case the
problem is specifically full-integer activation quantization for one or more layers and
`tf.lite.experimental.QuantizationDebugger` will name them. This API has moved around across
TF versions; if it errors out, checkpoint 3 (QAT) below works regardless of what's causing
the gap.

In [ ]:
note0 = results["0_float32"]["note_acc"]
note2 = results["2_full_int8"]["note_acc"]
f1_0 = results["0_float32"]["onset_f1"]
f1_2 = results["2_full_int8"]["onset_f1"]

STILL_DEGRADED = (note0 - note2) > 0.03 or (f1_0 - f1_2) > 0.05

if STILL_DEGRADED:
    print(f"Full-integer INT8 is still below the float baselines "
          f"(note {note0 * 100:.2f}% -> {note2 * 100:.2f}%, onset F1 {f1_0 * 100:.1f}% -> {f1_2 * 100:.1f}%).")
    print("Running the per-layer QuantizationDebugger to find which layer(s) are responsible.\n")
    try:
        debugger = tf.lite.experimental.QuantizationDebugger(
            converter=convert_int8(model, lambda: representative_data_gen(n_samples=200)),
            debug_dataset=lambda: representative_data_gen(n_samples=200),
        )
        debugger.run()
        layer_stats = debugger.layer_statistics_dump()
        if isinstance(layer_stats, pd.DataFrame):
            worst = layer_stats.sort_values("rmse/scale", ascending=False).head(15)
            print(worst.to_string())
        else:
            print(layer_stats)
    except Exception as e:
        print(f"QuantizationDebugger failed ({e!r}) -- this API varies across TF versions.")
        print("Checkpoint 3 (QAT) below sidesteps needing to identify the exact offending layer.")
else:
    print("Full-integer INT8 is within a reasonable margin of the float baselines.")
    print(f"  note accuracy  {note0 * 100:.2f}% -> {note2 * 100:.2f}%  ({(note2 - note0) * 100:+.2f})")
    print(f"  onset F1       {f1_0 * 100:.1f}% -> {f1_2 * 100:.1f}%  ({(f1_2 - f1_0) * 100:+.1f})")
    print("No per-layer debugging needed. Checkpoint 3 (QAT) below closes what's left.")

### Checkpoint 3 -- quantization-aware training (the deployment path)

PTQ takes a network that was trained in full precision, never asked to tolerate int8
rounding, and rounds it anyway -- whatever that costs, it costs after the fact, and
calibration can only work around it. QAT inserts fake-quantization ops into the forward pass
and fine-tunes, so the weights move to places where int8 rounding doesn't hurt in the first
place. The gradient still flows in float (straight-through estimator); only the forward pass
is discretized.

It is a fine-tune, not a retrain: a handful of epochs at a low learning rate, warm-started
from the trained weights.

**This step depends on the `Conv2D`/`DepthwiseConv2D` trunk chosen above.** `tfmot`'s 8-bit
scheme registers `QuantizeConfig`s for `Conv2D`, `DepthwiseConv2D`, `Dense`,
`BatchNormalization`, `ReLU`, pooling, `Flatten` and friends -- but not for
`DepthwiseConv1D`.

In [ ]:
print("=== Checkpoint 3: quantization-aware training (QAT) ===\n")

import tensorflow_model_optimization as tfmot

# `model` was built with `models.Model(...)` where `models = keras.models` from the tf_keras
# binding established in the setup cell -- this assertion catches a mismatch immediately
# (e.g. a kernel that still has an old, Keras-3-bound `model` sitting around after only a
# partial re-run) instead of surfacing as tfmot's much less obvious
# "`to_quantize` can only either be a keras Sequential or Functional model" error.
assert type(model).__module__.split(".")[0] in ("keras", "tf_keras"), (
    f"`model` is a {type(model)}, not built via the tf_keras binding from the setup cell. "
    "Restart the kernel and re-run the whole notebook from the top."
)
assert hasattr(model, "_is_graph_network") and model._is_graph_network, (
    "`model` doesn't look like a Keras 2 Functional model to tensorflow_model_optimization. "
    "See the Keras version note markdown cell near the top of the notebook."
)

qat_model = tfmot.quantization.keras.quantize_model(model)

# tfmot.quantize_model() prefixes every wrapped layer's name with "quant_", which renames
# this functional model's OUTPUTS too: "note_output"/"onset_output" become
# "quant_note_output"/"quant_onset_output". Nothing downstream of this line knows about that
# rename -- compile_model()'s dict-keyed losses/metrics, the tf.data pipelines' dict-keyed
# labels ({"note_output": ..., "onset_output": ...} from make_dataset()), and the
# EarlyStopping callback below (string monitor="val_note_output_accuracy") all still say
# "note_output"/"onset_output". Left alone this raises the moment .fit() runs, so the QAT
# model's output tensors are renamed back to match the original model's names.
# output_names is a plain list attribute on a Keras 2 / tf_keras functional model (not a
# computed property), so this is a metadata swap, not a graph change -- zero effect on the
# underlying quantized computation or on the .tflite this model converts to later.
try:
    qat_model.output_names = list(model.output_names)
except AttributeError:
    # Fallback for a tf_keras patch version where output_names isn't a plain settable
    # attribute: wrap each output in a no-op Activation("linear") layer carrying the
    # original name. Still a pure identity op.
    qat_model = keras.Model(
        inputs=qat_model.input,
        outputs=[layers.Activation("linear", name=name)(t)
                 for name, t in zip(model.output_names, qat_model.output)],
        name="guitar_midi_dscnn_qat",
    )

assert list(qat_model.output_names) == ["note_output", "onset_output"], qat_model.output_names
print(f"QAT model outputs renamed back to {list(qat_model.output_names)} (tfmot had "
      f"prefixed them with 'quant_').")

compile_model(qat_model, QAT_LEARNING_RATE)
print(f"QAT model built: {qat_model.count_params():,} params "
      f"(includes fake-quant range variables, so it exceeds the float model's count)\n")

qat_history = qat_model.fit(
    train_ds, validation_data=val_ds, epochs=QAT_EPOCHS,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_note_output_accuracy", mode="max",
                                      patience=3, restore_best_weights=True, verbose=1),
    ],
)

converter_qat = convert_int8(qat_model, representative_data_gen)
tflite_qat = converter_qat.convert()
qat_path = OUT_DIR / "checkpoint3_qat_int8.tflite"
qat_path.write_bytes(tflite_qat)
print(f"\nSaved: {qat_path} ({len(tflite_qat) / 1024:.1f} KB)")

raw_preds["3_qat_int8"] = collect_tflite_predictions(qat_path, test_files)
results["3_qat_int8"] = score_predictions(*raw_preds["3_qat_int8"], onset_threshold=ONSET_THRESHOLD)
print_scores("Checkpoint 3 -- QAT INT8", results["3_qat_int8"])

print(f"\nQAT vs. PTQ: {(results['3_qat_int8']['note_acc'] - results['2_full_int8']['note_acc']) * 100:+.2f} "
      f"points of note accuracy")
print(f"QAT vs. float32 baseline: "
      f"{(results['3_qat_int8']['note_acc'] - results['0_float32']['note_acc']) * 100:+.2f} points "
      f"<- this is the number that should be close to zero")

### Why there is no pruning stage

The obvious next lever after quantization is magnitude pruning. It is deliberately absent
here, for a concrete reason:

**TensorFlow Lite Micro has no sparse kernels.** `tfmot`'s magnitude pruning produces
*unstructured* sparsity -- individual weights zeroed, scattered arbitrarily through each
kernel. Converting that to a `.tflite` and running it under TFLM executes exactly the same
dense `CONV_2D` / `DEPTHWISE_CONV_2D` kernels, multiplying by the zeros like any other
weight. So on the ESP32-S3 it would buy:

- **no latency reduction** -- identical MAC count through esp-nn's dense int8 kernels
- **no RAM reduction** -- the tensor arena holds activations, which pruning doesn't touch
- **no flash reduction** -- the flatbuffer stores the zeros explicitly; only a sparsity-aware
  container format (which TFLM doesn't support here) would compress them

against a real, measurable accuracy cost. The model is well under the 150 KB flash budget, so
there is no size pressure to trade accuracy for.

Pruning would be worth revisiting only under one of two conditions: (1) flash pressure that
can't be solved by lowering `MODEL_WIDTH` -- and lowering `MODEL_WIDTH` gives *structured*
shrinkage, which does reduce real MACs and real RAM, making it strictly better than
unstructured pruning here; or (2) a runtime with sparse kernel support, which TFLM is not.

### Summary -- pick the best deployable (int8 in/out) checkpoint

Checkpoints 0 (float32) and 1 (dynamic-range, float activations) are diagnostic only: the
firmware's `quantize_and_load_input()` / `run_inference()` expect int8 in and int8 out, so
the deployed model must come from checkpoint 2 or 3.

Selection is on `note_acc_nonsilence + onset_f1` rather than `note_acc + onset_acc`, since the
latter two are both inflated by the silence/no-onset majority class and could prefer a model
that simply learned to stay quiet.

In [ ]:
print("=== Summary ===\n")
header = f"{'checkpoint':22s} {'note':>8s} {'note(ns)':>9s} {'onsetP':>8s} {'onsetR':>8s} {'onsetF1':>8s}"
print(header)
print("-" * len(header))
for k, s in results.items():
    print(f"{k:22s} {s['note_acc'] * 100:7.2f}% {s['note_acc_nonsilence'] * 100:8.2f}% "
          f"{s['onset_precision'] * 100:7.1f}% {s['onset_recall'] * 100:7.1f}% {s['onset_f1'] * 100:7.1f}%")

deployable_paths = {"2_full_int8": int8_path}
if "3_qat_int8" in results:
    deployable_paths["3_qat_int8"] = qat_path


def objective(k):
    s = results[k]
    return s["note_acc_nonsilence"] + s["onset_f1"]


best_key = max(deployable_paths, key=objective)
best_path = deployable_paths[best_key]
print(f"\nBest deployable (int8 in/out) checkpoint: {best_key}  ({best_path.name})")

final_model_path = OUT_DIR / "final_audio_model_int8.tflite"
final_model_path.write_bytes(best_path.read_bytes())
final_size = final_model_path.stat().st_size
print(f"Copied to {final_model_path} ({final_size / 1024:.1f} KB)")
if final_size > 150 * 1024:
    print(f"WARNING: over the 150 KB budget in the spec. Lower MODEL_WIDTH and re-run.")

# --- Accuracy ladder plot --------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 4))
keys = list(results.keys())
xs = np.arange(len(keys))
ax.bar(xs - 0.2, [results[k]["note_acc_nonsilence"] * 100 for k in keys], 0.4,
       color=CAT[0], label="note accuracy (non-silence frames)")
ax.bar(xs + 0.2, [results[k]["onset_f1"] * 100 for k in keys], 0.4,
       color=CAT[1], label="onset F1")
ax.axhline(90, color=MUTED, linestyle=":", linewidth=1)
ax.set_xticks(xs); ax.set_xticklabels(keys, rotation=20, ha="right")
ax.set_ylabel("%"); ax.set_ylim(0, 100)
ax.set_title("Accuracy at each stage -- flat bars mean quantization is costing nothing")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

### Confusion analysis for the deployed model

A confusion matrix concentrated on the immediate off-diagonals means the model has the right
pitch region and is missing by a semitone or two, which is an analysis-window / capacity
question. Mass in the silence row/column instead means the silence threshold or onset timing
is the issue, not pitch.

In [ ]:
t_notes, p_notes, t_onset, p_prob = raw_preds[best_key]

# Octave and semitone error breakdown, non-silence frames only.
mask = t_notes != SILENCE_INDEX
t_midi = t_notes[mask] + MIDI_LOW - 1
p_valid = p_notes[mask]
p_is_silence = p_valid == SILENCE_INDEX
p_midi = np.where(p_is_silence, -999, p_valid + MIDI_LOW - 1)
err = p_midi - t_midi

print(f"Non-silence frames: {mask.sum():,}")
print(f"  exact             : {np.mean(err == 0) * 100:6.2f}%")
print(f"  within +/-1 semitone: {np.mean(np.abs(err) <= 1) * 100:6.2f}%")
print(f"  octave errors     : {np.mean(np.isin(np.abs(err), [12, 24])) * 100:6.2f}%")
print(f"  predicted silence : {np.mean(p_is_silence) * 100:6.2f}%")
print(f"\nSilence frames predicted as a note (false note-ons): "
      f"{np.mean(p_notes[~mask] != SILENCE_INDEX) * 100:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

e = err[~p_is_silence]
bins = np.arange(-24.5, 25.5, 1)
axes[0].hist(np.clip(e, -24, 24), bins=bins, color=SEQ_BLUE[3], edgecolor=SURFACE, linewidth=0.4)
axes[0].set_yscale("log")
axes[0].axvline(0, color=CAT[7], linewidth=1)
for oct_err in (-12, 12):
    axes[0].axvline(oct_err, color=CAT[1], linestyle="--", linewidth=1)
axes[0].set_xlabel("prediction error (semitones)"); axes[0].set_ylabel("frames (log)")
axes[0].set_title("Pitch error distribution (dashed = octave errors)")

cm = np.zeros((num_classes, num_classes), dtype=np.int64)
np.add.at(cm, (t_notes, p_notes), 1)
cm_norm = cm / np.maximum(1, cm.sum(axis=1, keepdims=True))
im = axes[1].imshow(cm_norm, cmap="Blues", vmin=0, vmax=1, origin="lower", aspect="auto")
axes[1].set_xlabel("predicted class"); axes[1].set_ylabel("true class")
axes[1].set_title(f"Row-normalized confusion ({best_key})")
axes[1].grid(False)
fig.colorbar(im, ax=axes[1], fraction=0.046)

fig.tight_layout()
plt.show()

# Per-class recall, to find whether specific pitches are being abandoned entirely.
recall_per_class = np.diag(cm) / np.maximum(1, cm.sum(axis=1))
support = cm.sum(axis=1)
weak = [(class_names[i], int(support[i]), float(recall_per_class[i]))
        for i in range(num_classes) if support[i] >= 50 and recall_per_class[i] < 0.5]
if weak:
    print("\nClasses with >=50 test frames and <50% recall (candidates for more data or more capacity):")
    for name, sup, rec in sorted(weak, key=lambda t: t[2])[:20]:
        print(f"  class {name:>8s}  support {sup:6d}  recall {rec * 100:5.1f}%")
else:
    print("\nNo class with >=50 test frames falls below 50% recall.")

## Export C headers for the ESP32 firmware

`class_labels.h` carries every constant the firmware has to agree with this notebook on:
window size, hop size, sample rate, the normalization floor, the onset band width, the
validation-selected onset threshold, and the class table. The firmware checks for these at
compile time, so a mismatch between the two files becomes a compile error instead of a silent
accuracy loss that looks like a quantization problem.

In [ ]:
def write_c_header(tflite_file: pathlib.Path, header_file: pathlib.Path, array_name: str = "model") -> None:
    model_bytes = tflite_file.read_bytes()
    lines = [
        "#pragma once",
        "",
        f"// Generated by train_guitar_midi.ipynb -- {tflite_file.name}, {len(model_bytes):,} bytes",
        "",
        f"alignas(16) const unsigned char {array_name}[] = {{",
    ]
    for start in range(0, len(model_bytes), 12):
        chunk = model_bytes[start:start + 12]
        lines.append("  " + ", ".join(f"0x{b:02x}" for b in chunk) + ",")
    lines.extend(["};", f"const unsigned int {array_name}_len = {len(model_bytes)};", ""])
    header_file.write_text("\n".join(lines), encoding="utf-8")


header_path = OUT_DIR / "model.h"
write_c_header(final_model_path, header_path, array_name="model")
print(f"Saved: {header_path} ({header_path.stat().st_size:,} bytes)")
print("  (alignas(16) matters: TFLite Micro requires the flatbuffer to be 16-byte aligned,")
print("   and without it you get an alignment fault or silent misbehaviour on some builds.)")


def write_class_labels_header(class_names, header_file: pathlib.Path) -> None:
    entries = ["-1" if name == "silence" else str(int(name)) for name in class_names]
    lines = [
        "#pragma once",
        "",
        "// Generated by train_guitar_midi.ipynb. Do not edit by hand.",
        "//",
        "// Every constant here must agree with the training pipeline. guitar_midi_esp32s3.ino",
        "// checks for them at compile time, so a mismatch is a compile error rather than a",
        "// silent train/deploy input-distribution mismatch (which presents as a mysterious",
        "// accuracy loss that looks like, but is not, a quantization problem).",
        "",
        "// --- Framing (must match the .ino) ---",
        f"#define MODEL_WINDOW_SAMPLES {WINDOW_SAMPLES}",
        f"#define MODEL_HOP_SAMPLES    {HOP_SAMPLES}",
        f"#define MODEL_SAMPLE_RATE    {SAMPLE_RATE}",
        "",
        "// --- Window normalization ---",
        "// The firmware divides each zero-meaned window by max(peak, MODEL_NORM_PEAK_FLOOR).",
        f"// {NORM_PEAK_FLOOR_DBFS:.0f} dBFS peak, ~10 dB above the {SILENCE_RMS_DBFS:.0f} dBFS RMS silence gate.",
        f"#define MODEL_NORM_PEAK_FLOOR {NORM_PEAK_FLOOR:.6f}f",
        "",
        "// --- Onset timing ---",
        "// Width of the trailing band (ms) in which a real attack makes a window onset-positive.",
        "// Drives the firmware's RETRIGGER_LOCKOUT_MS (onset band + one inference period).",
        f"#define MODEL_ONSET_WINDOW_MS {ONSET_WINDOW_MS:.0f}",
        "",
        "// --- Onset decision threshold (swept on the validation split, not on test) ---",
        f"#define MODEL_ONSET_THRESHOLD {ONSET_THRESHOLD:.2f}f",
        "",
        "// --- Note classes ---",
        '// -1 marks the "silence" class. Every other entry is a MIDI note number (0-127),',
        "// in the same order the note-head softmax uses.",
        f"const int NUM_NOTE_CLASSES = {len(class_names)};",
        f"const int8_t CLASS_TO_MIDI[NUM_NOTE_CLASSES] = {{{', '.join(entries)}}};",
        "",
    ]
    header_file.write_text("\n".join(lines), encoding="utf-8")


class_labels_path = OUT_DIR / "class_labels.h"
write_class_labels_header(class_names, class_labels_path)
print(f"\nSaved: {class_labels_path}")
print(class_labels_path.read_text())

# --- Machine-readable run summary, for the report ---------------------------
summary = {
    "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
    "config": {
        "window_samples": WINDOW_SAMPLES, "hop_samples": HOP_SAMPLES,
        "sample_rate": SAMPLE_RATE, "stem_stride": STEM_STRIDE,
        "num_classes": num_classes, "midi_low": MIDI_LOW, "midi_high": MIDI_HIGH,
        "model_width": MODEL_WIDTH, "params": int(total_params), "macs": int(macs),
        "norm_peak_floor_dbfs": NORM_PEAK_FLOOR_DBFS,
        "onset_window_ms": ONSET_WINDOW_MS,
        "onset_threshold": ONSET_THRESHOLD,
        "test_players": sorted(TEST_PLAYERS),
    },
    "keras_evaluate": {k: float(v) for k, v in eval_results.items()},
    "checkpoints": {k: {kk: (float(vv) if isinstance(vv, (int, float)) else vv)
                        for kk, vv in s.items()} for k, s in results.items()},
    "deployed": {"checkpoint": best_key, "bytes": int(final_size)},
}
(OUT_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2))
print(f"\nSaved: {OUT_DIR / 'run_summary.json'}")

print("\nDone. Copy model.h and class_labels.h from "
      f"{OUT_DIR}/ into your Arduino sketch folder next to guitar_midi_esp32s3.ino.")